<a href="https://colab.research.google.com/github/ArghyaRC96/pricegraph-edge/blob/main/notebooks/02_pricegraph_final_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PriceGraph Edge — Final Pipeline

## 1. Repository Setup

This notebook contains the clean, reproducible pipeline for **PriceGraph Edge**.

The project repository is cloned from GitHub at the start of every fresh Colab session.  
Git LFS is enabled so the Walmart M5 raw datasets stored in the repository can be retrieved correctly.

> **Notebook 01:** EDA, experimentation, model comparison, and diagnostics.  
> **Notebook 02:** Final modelling, evaluation, pricing simulation, and revenue optimization.

In [4]:
# Install Git LFS for the large M5 CSV files
!apt-get -qq update
!apt-get -qq install git-lfs

# Enable Git LFS
!git lfs install

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package git-lfs.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.0.2-1ubuntu0.3_amd64.deb ...
Unpacking git-lfs (3.0.2-1ubuntu0.3) ...
Setting up git-lfs (3.0.2-1ubuntu0.3) ...
Processing triggers for man-db (2.10.2-1) ...
Git LFS initialized.


In [5]:
from pathlib import Path
import os

REPO_URL = "https://github.com/ArghyaRC96/pricegraph-edge.git"
PROJECT_ROOT = Path("/content/pricegraph-edge")

# Clone only if the repository is not already present
if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print("Repository already exists.")

# Move notebook runtime into the project repository
os.chdir(PROJECT_ROOT)

print("Current directory:")
print(Path.cwd())

Cloning into '/content/pricegraph-edge'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 11 (delta 0), reused 11 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (11/11), done.
Filtering content: 100% (3/3), 310.17 MiB | 4.83 MiB/s, done.
Current directory:
/content/pricegraph-edge


In [6]:
!git lfs pull

print("Git LFS files pulled.")

Git LFS files pulled.


In [7]:
print("Project folders:\n")

for path in sorted(PROJECT_ROOT.iterdir()):
    print("-", path.name)

print("\nRaw data files:\n")

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"

for path in sorted(RAW_DATA_PATH.glob("*")):
    size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"{path.name:<35}"
        f"{size_mb:>10.2f} MB"
    )

Project folders:

- .git
- .gitattributes
- .gitignore
- README.md
- data
- docs
- requirements.txt

Raw data files:

calendar.csv                             0.10 MB
sales_train_evaluation.csv             116.10 MB
sell_prices.csv                        193.97 MB


## 2. Imports & Configuration

This section imports the libraries required for data preparation, feature engineering, model training, evaluation, and pricing simulation.

It also defines the project paths and global configuration used throughout the notebook.

In [8]:
from pathlib import Path
import warnings
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

RANDOM_STATE = 42

print("Imports complete.")

Imports complete.


In [9]:
PROJECT_ROOT = Path("/content/pricegraph-edge")

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

MODEL_PATH = PROJECT_ROOT / "models"
OUTPUT_PATH = PROJECT_ROOT / "outputs"

PROCESSED_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Project Root :", PROJECT_ROOT)
print("Raw Data     :", RAW_DATA_PATH)
print("Processed    :", PROCESSED_DATA_PATH)
print("Models       :", MODEL_PATH)
print("Outputs      :", OUTPUT_PATH)

Project Root : /content/pricegraph-edge
Raw Data     : /content/pricegraph-edge/data/raw
Processed    : /content/pricegraph-edge/data/processed
Models       : /content/pricegraph-edge/models
Outputs      : /content/pricegraph-edge/outputs


In [10]:
import sklearn
import xgboost

print("Library Versions\n")

print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost     :", xgboost.__version__)

Library Versions

NumPy       : 2.1.3
Pandas      : 2.2.3
Scikit-learn: 1.6.1
XGBoost     : 3.4.1


## 3. Load & Validate M5 Data

The pipeline uses the three core Walmart M5 datasets:

- `calendar.csv` — dates, events, and SNAP indicators.
- `sell_prices.csv` — weekly SKU-store selling prices.
- `sales_train_evaluation.csv` — daily unit sales for every SKU-store combination.

This section loads the raw files directly from the cloned repository and validates their structure before any processing begins.

In [11]:
required_files = {
    "calendar": RAW_DATA_PATH / "calendar.csv",
    "sell_prices": RAW_DATA_PATH / "sell_prices.csv",
    "sales": RAW_DATA_PATH / "sales_train_evaluation.csv"
}

print("RAW FILE CHECK\n")

for name, path in required_files.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )

    size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"{name:<15} "
        f"FOUND | {size_mb:,.2f} MB"
    )

RAW FILE CHECK

calendar        FOUND | 0.10 MB
sell_prices     FOUND | 193.97 MB
sales           FOUND | 116.10 MB


In [12]:
calendar = pd.read_csv(
    required_files["calendar"],
    parse_dates=["date"]
)

sell_prices = pd.read_csv(
    required_files["sell_prices"]
)

sales = pd.read_csv(
    required_files["sales"]
)

print("Datasets loaded successfully.\n")

print("Calendar    :", calendar.shape)
print("Sell Prices :", sell_prices.shape)
print("Sales       :", sales.shape)

Datasets loaded successfully.

Calendar    : (1969, 14)
Sell Prices : (6841121, 4)
Sales       : (30490, 1947)


In [13]:
required_calendar_cols = {
    "date",
    "wm_yr_wk",
    "d",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
}

required_price_cols = {
    "store_id",
    "item_id",
    "wm_yr_wk",
    "sell_price"
}

required_sales_cols = {
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
}

validation_checks = {
    "Calendar":
        required_calendar_cols
        - set(calendar.columns),

    "Sell Prices":
        required_price_cols
        - set(sell_prices.columns),

    "Sales":
        required_sales_cols
        - set(sales.columns)
}

print("COLUMN VALIDATION\n")

for dataset, missing_cols in validation_checks.items():

    if missing_cols:
        print(
            f"{dataset:<12}: "
            f"MISSING {sorted(missing_cols)}"
        )

    else:
        print(
            f"{dataset:<12}: OK"
        )

COLUMN VALIDATION

Calendar    : OK
Sell Prices : OK
Sales       : OK


In [14]:
day_columns = [
    col
    for col in sales.columns
    if col.startswith("d_")
]

print("DATA INTEGRITY\n")

print(
    "Daily sales columns :",
    len(day_columns)
)

print(
    "Unique products     :",
    sales["item_id"].nunique()
)

print(
    "Unique stores       :",
    sales["store_id"].nunique()
)

print(
    "Unique states       :",
    sales["state_id"].nunique()
)

print(
    "Categories          :",
    sorted(
        sales["cat_id"].unique()
    )
)

print(
    "Calendar range      :",
    calendar["date"].min().date(),
    "to",
    calendar["date"].max().date()
)

print(
    "Missing sell prices :",
    sell_prices["sell_price"]
    .isna()
    .sum()
)

DATA INTEGRITY

Daily sales columns : 1941
Unique products     : 3049
Unique stores       : 10
Unique states       : 3
Categories          : ['FOODS', 'HOBBIES', 'HOUSEHOLD']
Calendar range      : 2011-01-29 to 2016-06-19
Missing sell prices : 0


## 4. Select Pricing-Suitable SKUs

PriceGraph Edge focuses on a curated portfolio of **100 FOODS products** with sufficient demand activity, pricing variation, and historical coverage for pricing analysis.

A pricing suitability score is calculated using:

- total unit sales,
- active sales ratio,
- number of unique prices,
- historical price changes,
- weekly price coverage.

To maintain representation across departments, the final portfolio contains:

- **20 products from FOODS_1**
- **30 products from FOODS_2**
- **50 products from FOODS_3**

This portfolio selection is used consistently throughout the modelling and pricing pipeline.

In [15]:
foods_sales = sales[
    sales["cat_id"] == "FOODS"
].copy()

day_columns = [
    col
    for col in foods_sales.columns
    if col.startswith("d_")
]

foods_sales["_total_units"] = (
    foods_sales[day_columns]
    .sum(axis=1)
)

foods_sales["_active_store_days"] = (
    foods_sales[day_columns]
    .gt(0)
    .sum(axis=1)
)

sales_metrics = (
    foods_sales
    .groupby(
        ["item_id", "dept_id"],
        as_index=False
    )
    .agg(
        total_units=(
            "_total_units",
            "sum"
        ),
        active_store_days=(
            "_active_store_days",
            "sum"
        ),
        stores_present=(
            "store_id",
            "nunique"
        )
    )
)

sales_metrics["active_sales_ratio"] = (
    sales_metrics["active_store_days"]
    /
    (
        len(day_columns)
        *
        sales_metrics["stores_present"]
    )
)

print(
    "FOODS products:",
    sales_metrics["item_id"].nunique()
)

print(
    "\nDepartment counts:\n",
    sales_metrics["dept_id"]
    .value_counts()
    .sort_index()
    .to_string()
)

FOODS products: 1437

Department counts:
 dept_id
FOODS_1    216
FOODS_2    398
FOODS_3    823


In [16]:
foods_item_ids = set(
    sales_metrics["item_id"]
)

foods_prices = sell_prices[
    sell_prices["item_id"].isin(
        foods_item_ids
    )
].copy()

foods_prices = (
    foods_prices
    .sort_values(
        [
            "item_id",
            "store_id",
            "wm_yr_wk"
        ]
    )
    .reset_index(drop=True)
)

price_difference = (
    foods_prices
    .groupby(
        ["item_id", "store_id"]
    )["sell_price"]
    .diff()
)

foods_prices["price_changed"] = (
    price_difference.notna()
    &
    price_difference.ne(0)
).astype(int)

price_metrics = (
    foods_prices
    .groupby(
        "item_id",
        as_index=False
    )
    .agg(
        unique_prices=(
            "sell_price",
            "nunique"
        ),
        price_changes=(
            "price_changed",
            "sum"
        ),
        average_price=(
            "sell_price",
            "mean"
        ),
        price_std=(
            "sell_price",
            "std"
        ),
        weeks_available=(
            "wm_yr_wk",
            "nunique"
        ),
        stores_priced=(
            "store_id",
            "nunique"
        )
    )
)

total_price_weeks = (
    sell_prices["wm_yr_wk"]
    .nunique()
)

total_stores = (
    foods_sales["store_id"]
    .nunique()
)

price_metrics["week_coverage"] = (
    price_metrics["weeks_available"]
    /
    total_price_weeks
)

price_metrics["store_coverage"] = (
    price_metrics["stores_priced"]
    /
    total_stores
)

print(
    "Price metrics created for:",
    price_metrics["item_id"].nunique(),
    "products"
)

Price metrics created for: 1437 products


In [17]:
foods_suitability = (
    sales_metrics
    .merge(
        price_metrics,
        on="item_id",
        how="inner"
    )
)

score_columns = [
    "total_units",
    "active_sales_ratio",
    "unique_prices",
    "price_changes",
    "week_coverage"
]

for col in score_columns:

    foods_suitability[
        f"{col}_score"
    ] = (
        foods_suitability[col]
        .rank(
            pct=True,
            method="average"
        )
    )

foods_suitability["suitability_score"] = (
    0.25
    * foods_suitability["total_units_score"]

    + 0.20
    * foods_suitability["active_sales_ratio_score"]

    + 0.15
    * foods_suitability["unique_prices_score"]

    + 0.30
    * foods_suitability["price_changes_score"]

    + 0.10
    * foods_suitability["week_coverage_score"]
)

print(
    foods_suitability[
        [
            "total_units",
            "active_sales_ratio",
            "unique_prices",
            "price_changes",
            "week_coverage",
            "suitability_score"
        ]
    ]
    .describe()
    .round(3)
    .to_string()
)

       total_units  active_sales_ratio  unique_prices  price_changes  week_coverage  suitability_score
count     1437.000            1437.000       1437.000       1437.000       1437.000           1437.000
mean     31957.152               0.382          6.974         35.808          0.831              0.500
std      60366.558               0.206          5.862         32.351          0.226              0.214
min        885.000               0.038          1.000          0.000          0.266              0.016
25%       7518.000               0.215          3.000         12.000          0.656              0.341
50%      15351.000               0.359          6.000         26.000          0.996              0.522
75%      30343.000               0.523          9.000         51.000          1.000              0.658
max    1017916.000               0.995         52.000        198.000          1.000              0.963


In [18]:
department_allocation = {
    "FOODS_1": 20,
    "FOODS_2": 30,
    "FOODS_3": 50
}

selected_parts = []

for dept_id, product_count in department_allocation.items():

    department_products = (
        foods_suitability[
            foods_suitability["dept_id"]
            == dept_id
        ]
        .sort_values(
            [
                "suitability_score",
                "item_id"
            ],
            ascending=[
                False,
                True
            ]
        )
        .head(product_count)
    )

    selected_parts.append(
        department_products
    )

selected_products = (
    pd.concat(
        selected_parts,
        ignore_index=True
    )
)

selected_item_ids = (
    selected_products["item_id"]
    .tolist()
)

print(
    "Selected products:",
    len(selected_item_ids)
)

print(
    "\nSelected by department:\n",
    selected_products["dept_id"]
    .value_counts()
    .sort_index()
    .to_string()
)

Selected products: 100

Selected by department:
 dept_id
FOODS_1    20
FOODS_2    30
FOODS_3    50


In [19]:
expected_selection = {
    "FOODS_1": 20,
    "FOODS_2": 30,
    "FOODS_3": 50
}

actual_selection = (
    selected_products["dept_id"]
    .value_counts()
    .to_dict()
)

assert (
    selected_products["item_id"]
    .nunique()
    == 100
), "Expected exactly 100 unique products."

assert (
    actual_selection
    == expected_selection
), "Department allocation does not match the locked portfolio."

SELECTED_SKU_PATH = (
    PROCESSED_DATA_PATH
    / "selected_pricing_skus.csv"
)

selected_products.to_csv(
    SELECTED_SKU_PATH,
    index=False
)

print("Portfolio validation: PASSED")
print("Saved:", SELECTED_SKU_PATH)

Portfolio validation: PASSED
Saved: /content/pricegraph-edge/data/processed/selected_pricing_skus.csv


## 5. Build Daily Pricing Dataset

The selected 100-product portfolio is converted from the original wide M5 sales format into a daily SKU-store panel.

The daily sales data is then enriched with:

- calendar and week identifiers,
- state-specific SNAP indicators,
- event information,
- event-type flags,
- weekly Walmart selling prices.

Rows without an observed selling price are excluded from the pricing dataset because no valid price-demand relationship can be evaluated for those observations.

In [20]:
selected_sales_wide = sales[
    sales["item_id"].isin(selected_item_ids)
].copy()

print("Selected wide rows:", len(selected_sales_wide))

print(
    "Unique products:",
    selected_sales_wide["item_id"].nunique()
)

print(
    "Unique stores:",
    selected_sales_wide["store_id"].nunique()
)

print(
    "Product-store combinations:",
    selected_sales_wide[
        ["item_id", "store_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

Selected wide rows: 1000
Unique products: 100
Unique stores: 10
Product-store combinations: 1000


In [21]:
sales_long = selected_sales_wide.melt(
    id_vars=[
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ],
    value_vars=day_columns,
    var_name="d",
    value_name="units_sold"
)

print("Long rows:", len(sales_long))

print(
    "Expected rows:",
    100 * 10 * len(day_columns)
)

print(
    "Unique products:",
    sales_long["item_id"].nunique()
)

print(
    "Unique stores:",
    sales_long["store_id"].nunique()
)

Long rows: 1941000
Expected rows: 1941000
Unique products: 100
Unique stores: 10


In [22]:
assert len(sales_long) == 1_941_000, (
    "Unexpected long-format row count."
)

assert sales_long["item_id"].nunique() == 100
assert sales_long["store_id"].nunique() == 10

print("Long-format validation: PASSED")

Long-format validation: PASSED


In [23]:
calendar_features = calendar[
    [
        "d",
        "date",
        "wm_yr_wk",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2",
        "snap_CA",
        "snap_TX",
        "snap_WI"
    ]
].copy()

daily_data = sales_long.merge(
    calendar_features,
    on="d",
    how="left",
    validate="many_to_one"
)

print("Rows after calendar merge:", len(daily_data))

print(
    "Missing dates:",
    daily_data["date"]
    .isna()
    .sum()
)

Rows after calendar merge: 1941000
Missing dates: 0


In [24]:
daily_data["snap"] = np.select(
    [
        daily_data["state_id"].eq("CA"),
        daily_data["state_id"].eq("TX"),
        daily_data["state_id"].eq("WI")
    ],
    [
        daily_data["snap_CA"],
        daily_data["snap_TX"],
        daily_data["snap_WI"]
    ],
    default=0
).astype(int)

daily_data["event_flag"] = (
    daily_data["event_name_1"].notna()
    |
    daily_data["event_name_2"].notna()
).astype(int)

In [25]:
event_types = {
    "sporting_flag": "Sporting",
    "cultural_flag": "Cultural",
    "religious_flag": "Religious",
    "national_flag": "National"
}

for feature_name, event_type in event_types.items():

    daily_data[feature_name] = (
        daily_data["event_type_1"].eq(event_type)
        |
        daily_data["event_type_2"].eq(event_type)
    ).astype(int)

print(
    daily_data[
        [
            "snap",
            "event_flag",
            "sporting_flag",
            "cultural_flag",
            "religious_flag",
            "national_flag"
        ]
    ]
    .sum()
    .to_string()
)

snap              640000
event_flag        158000
sporting_flag      16000
cultural_flag      40000
religious_flag     55000
national_flag      51000


In [26]:
daily_data = daily_data.merge(
    sell_prices[
        [
            "store_id",
            "item_id",
            "wm_yr_wk",
            "sell_price"
        ]
    ],
    on=[
        "store_id",
        "item_id",
        "wm_yr_wk"
    ],
    how="left",
    validate="many_to_one"
)

price_coverage = (
    daily_data["sell_price"]
    .notna()
    .mean()
    * 100
)

missing_price_rows = (
    daily_data["sell_price"]
    .isna()
    .sum()
)

print(
    f"Price coverage: {price_coverage:.2f}%"
)

print(
    "Rows without price:",
    missing_price_rows
)

print(
    "Rows with price:",
    daily_data["sell_price"]
    .notna()
    .sum()
)

Price coverage: 97.50%
Rows without price: 48545
Rows with price: 1892455


In [27]:
pricing_data = (
    daily_data[
        daily_data["sell_price"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

print("DAILY PRICING DATASET\n")

print("Rows       :", len(pricing_data))
print(
    "Products   :",
    pricing_data["item_id"].nunique()
)
print(
    "Stores     :",
    pricing_data["store_id"].nunique()
)
print(
    "Departments:",
    pricing_data["dept_id"].nunique()
)

print(
    "Date range :",
    pricing_data["date"].min().date(),
    "to",
    pricing_data["date"].max().date()
)

print(
    "Average price:",
    round(
        pricing_data["sell_price"].mean(),
        2
    )
)

print(
    "Total units:",
    int(
        pricing_data["units_sold"].sum()
    )
)

DAILY PRICING DATASET

Rows       : 1892455
Products   : 100
Stores     : 10
Departments: 3
Date range : 2011-01-29 to 2016-05-22
Average price: 2.95
Total units: 9682933


In [28]:
DAILY_PRICING_PATH = (
    PROCESSED_DATA_PATH
    / "daily_pricing_data.csv"
)

pricing_data.to_csv(
    DAILY_PRICING_PATH,
    index=False
)

print(
    "Saved:",
    DAILY_PRICING_PATH
)

Saved: /content/pricegraph-edge/data/processed/daily_pricing_data.csv


## 6. Build Weekly Modelling Dataset

Walmart M5 selling prices are recorded at a **weekly** level, so the final demand model also operates at the SKU-store-week grain.

Daily observations are aggregated into weekly records containing:

- total units sold,
- weekly selling price,
- SNAP-active days,
- event days,
- event-type days,
- weekly revenue.

This produces one row per product, store, and Walmart week.

In [29]:
weekly_data = (
    pricing_data
    .groupby(
        [
            "item_id",
            "dept_id",
            "store_id",
            "state_id",
            "wm_yr_wk"
        ],
        as_index=False
    )
    .agg(
        units_sold=(
            "units_sold",
            "sum"
        ),
        sell_price=(
            "sell_price",
            "first"
        ),
        date=(
            "date",
            "min"
        ),
        snap_days=(
            "snap",
            "sum"
        ),
        event_days=(
            "event_flag",
            "sum"
        ),
        sporting_days=(
            "sporting_flag",
            "sum"
        ),
        cultural_days=(
            "cultural_flag",
            "sum"
        ),
        religious_days=(
            "religious_flag",
            "sum"
        ),
        national_days=(
            "national_flag",
            "sum"
        )
    )
)

weekly_data = (
    weekly_data
    .sort_values(
        [
            "item_id",
            "store_id",
            "date"
        ]
    )
    .reset_index(drop=True)
)

print("Weekly rows:", len(weekly_data))

print(
    "Products:",
    weekly_data["item_id"].nunique()
)

print(
    "Stores:",
    weekly_data["store_id"].nunique()
)

print(
    "Unique weeks:",
    weekly_data["wm_yr_wk"].nunique()
)

Weekly rows: 271065
Products: 100
Stores: 10
Unique weeks: 278


In [30]:
weekly_data["revenue"] = (
    weekly_data["sell_price"]
    *
    weekly_data["units_sold"]
)

print(
    weekly_data[
        [
            "units_sold",
            "sell_price",
            "revenue"
        ]
    ]
    .describe()
    .round(2)
    .to_string()
)

       units_sold  sell_price    revenue
count   271065.00   271065.00  271065.00
mean        35.72        2.95      75.17
std         71.92        2.27     127.01
min          0.00        0.05       0.00
25%          5.00        1.08      12.22
50%         17.00        2.24      38.36
75%         39.00        4.12      86.90
max       3539.00       13.98    3539.00


In [31]:
weekly_data["month"] = (
    weekly_data["date"]
    .dt.month
)

weekly_data["year"] = (
    weekly_data["date"]
    .dt.year
)

weekly_data["quarter"] = (
    weekly_data["date"]
    .dt.quarter
)

weekly_data["week_of_year"] = (
    weekly_data["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

weekly_data["day_of_month"] = (
    weekly_data["date"]
    .dt.day
)

In [32]:
weekly_data["sporting_flag"] = (
    weekly_data["sporting_days"] > 0
).astype(int)

weekly_data["cultural_flag"] = (
    weekly_data["cultural_days"] > 0
).astype(int)

weekly_data["religious_flag"] = (
    weekly_data["religious_days"] > 0
).astype(int)

weekly_data["national_flag"] = (
    weekly_data["national_days"] > 0
).astype(int)

print(
    weekly_data[
        [
            "sporting_flag",
            "cultural_flag",
            "religious_flag",
            "national_flag"
        ]
    ]
    .sum()
    .to_string()
)

sporting_flag     15482
cultural_flag     38869
religious_flag    48763
national_flag     49904


In [33]:
duplicate_weekly_rows = (
    weekly_data[
        [
            "item_id",
            "store_id",
            "wm_yr_wk"
        ]
    ]
    .duplicated()
    .sum()
)

print("WEEKLY DATA CHECK\n")

print(
    "Rows:",
    len(weekly_data)
)

print(
    "Duplicate SKU-store-week rows:",
    duplicate_weekly_rows
)

print(
    "Missing prices:",
    weekly_data["sell_price"]
    .isna()
    .sum()
)

print(
    "Missing demand:",
    weekly_data["units_sold"]
    .isna()
    .sum()
)

print(
    "Date range:",
    weekly_data["date"].min().date(),
    "to",
    weekly_data["date"].max().date()
)

WEEKLY DATA CHECK

Rows: 271065
Duplicate SKU-store-week rows: 0
Missing prices: 0
Missing demand: 0
Date range: 2011-01-29 to 2016-05-21


In [34]:
WEEKLY_DATA_PATH = (
    PROCESSED_DATA_PATH
    / "weekly_pricing_data.csv"
)

weekly_data.to_csv(
    WEEKLY_DATA_PATH,
    index=False
)

print(
    "Saved:",
    WEEKLY_DATA_PATH
)

Saved: /content/pricegraph-edge/data/processed/weekly_pricing_data.csv


## 7. Leakage-Safe Feature Engineering

The demand model uses historical pricing, demand, seasonality, event, and product-store context.

All historical features are created using only information available **before the week being predicted**. This is enforced using `shift(1)` before rolling or expanding calculations.

The final feature groups include:

- price and relative-price features,
- lagged demand,
- rolling demand history,
- SKU-store historical demand normalization,
- SNAP and event indicators,
- calendar seasonality,
- product, department, store, and state identifiers.

The target variable is weekly `units_sold`.

In [35]:
feature_data = (
    weekly_data
    .sort_values(
        [
            "item_id",
            "store_id",
            "date"
        ]
    )
    .reset_index(drop=True)
    .copy()
)

group_keys = [
    "item_id",
    "store_id"
]

print("Feature engineering rows:", len(feature_data))

Feature engineering rows: 271065


In [36]:
feature_data["previous_price"] = (
    feature_data
    .groupby(group_keys)["sell_price"]
    .shift(1)
)

feature_data["normal_price_13wk"] = (
    feature_data
    .groupby(group_keys)["sell_price"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=13,
            min_periods=4
        )
        .median()
    )
)

feature_data["hist_mean_price"] = (
    feature_data
    .groupby(group_keys)["sell_price"]
    .transform(
        lambda x:
        x.shift(1)
        .expanding(
            min_periods=4
        )
        .mean()
    )
)

In [37]:
feature_data["price_change_pct"] = (
    (
        feature_data["sell_price"]
        - feature_data["previous_price"]
    )
    /
    feature_data["previous_price"]
) * 100

feature_data["price_vs_normal_pct"] = (
    (
        feature_data["sell_price"]
        - feature_data["normal_price_13wk"]
    )
    /
    feature_data["normal_price_13wk"]
) * 100

feature_data["price_vs_hist_mean_pct"] = (
    (
        feature_data["sell_price"]
        - feature_data["hist_mean_price"]
    )
    /
    feature_data["hist_mean_price"]
) * 100

feature_data["price_down_flag"] = (
    feature_data["sell_price"]
    <
    feature_data["previous_price"]
).astype(int)

feature_data["price_up_flag"] = (
    feature_data["sell_price"]
    >
    feature_data["previous_price"]
).astype(int)

In [38]:
for lag in [1, 4, 13, 52]:

    feature_data[
        f"lag_{lag}_units"
    ] = (
        feature_data
        .groupby(group_keys)["units_sold"]
        .shift(lag)
    )

In [39]:
feature_data["rolling_4_units"] = (
    feature_data
    .groupby(group_keys)["units_sold"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=4,
            min_periods=4
        )
        .mean()
    )
)

feature_data["rolling_13_units"] = (
    feature_data
    .groupby(group_keys)["units_sold"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=13,
            min_periods=4
        )
        .mean()
    )
)

In [40]:
feature_data["hist_mean_units"] = (
    feature_data
    .groupby(group_keys)["units_sold"]
    .transform(
        lambda x:
        x.shift(1)
        .expanding(
            min_periods=4
        )
        .mean()
    )
)

feature_data["hist_median_units"] = (
    feature_data
    .groupby(group_keys)["units_sold"]
    .transform(
        lambda x:
        x.shift(1)
        .expanding(
            min_periods=4
        )
        .median()
    )
)

In [41]:
feature_data["lag_1_vs_mean"] = (
    feature_data["lag_1_units"]
    /
    feature_data["hist_mean_units"]
)

feature_data["lag_4_vs_mean"] = (
    feature_data["lag_4_units"]
    /
    feature_data["hist_mean_units"]
)

feature_data["lag_13_vs_mean"] = (
    feature_data["lag_13_units"]
    /
    feature_data["hist_mean_units"]
)

feature_data["lag_52_vs_mean"] = (
    feature_data["lag_52_units"]
    /
    feature_data["hist_mean_units"]
)

feature_data["rolling_4_vs_mean"] = (
    feature_data["rolling_4_units"]
    /
    feature_data["hist_mean_units"]
)

feature_data["rolling_13_vs_mean"] = (
    feature_data["rolling_13_units"]
    /
    feature_data["hist_mean_units"]
)

In [42]:
feature_data["month"] = (
    feature_data["date"]
    .dt.month
)

feature_data["day_of_month"] = (
    feature_data["date"]
    .dt.day
)

feature_data["week_of_year"] = (
    feature_data["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

In [43]:
categorical_features = [
    "item_id",
    "dept_id",
    "store_id",
    "state_id"
]

numeric_features = [
    # Price
    "sell_price",
    "price_change_pct",
    "price_vs_normal_pct",
    "price_vs_hist_mean_pct",
    "price_down_flag",
    "price_up_flag",

    # Raw demand history
    "lag_1_units",
    "lag_4_units",
    "lag_13_units",
    "lag_52_units",
    "rolling_4_units",
    "rolling_13_units",

    # Normalized demand history
    "hist_mean_units",
    "hist_median_units",
    "lag_1_vs_mean",
    "lag_4_vs_mean",
    "lag_13_vs_mean",
    "lag_52_vs_mean",
    "rolling_4_vs_mean",
    "rolling_13_vs_mean",

    # Events
    "snap_days",
    "event_days",
    "sporting_flag",
    "cultural_flag",
    "religious_flag",
    "national_flag",

    # Calendar
    "month",
    "day_of_month",
    "week_of_year"
]

champion_features = (
    categorical_features
    + numeric_features
)

TARGET = "units_sold"

print("Categorical features:", len(categorical_features))
print("Numeric features    :", len(numeric_features))
print("Total features      :", len(champion_features))

Categorical features: 4
Numeric features    : 29
Total features      : 33


In [44]:
model_data = (
    feature_data
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna(
        subset=champion_features
    )
    .copy()
    .reset_index(drop=True)
)

print("MODEL DATASET\n")

print("Rows       :", len(model_data))
print(
    "Products   :",
    model_data["item_id"].nunique()
)
print(
    "Stores     :",
    model_data["store_id"].nunique()
)

print(
    "Date range :",
    model_data["date"].min().date(),
    "to",
    model_data["date"].max().date()
)

print(
    "Feature missing values:",
    model_data[
        champion_features
    ]
    .isna()
    .sum()
    .sum()
)

MODEL DATASET

Rows       : 219065
Products   : 100
Stores     : 10
Date range : 2012-01-28 to 2016-05-21
Feature missing values: 0


In [45]:
print("FEATURE HEALTH CHECK\n")

print(
    "Duplicate SKU-store-week rows:",
    model_data[
        [
            "item_id",
            "store_id",
            "wm_yr_wk"
        ]
    ]
    .duplicated()
    .sum()
)

print(
    "Negative demand rows:",
    (
        model_data[TARGET] < 0
    ).sum()
)

print(
    "Infinite numeric values:",
    np.isinf(
        model_data[numeric_features]
        .to_numpy()
    ).sum()
)

print(
    "Missing model values:",
    model_data[
        champion_features
        + [TARGET]
    ]
    .isna()
    .sum()
    .sum()
)

FEATURE HEALTH CHECK

Duplicate SKU-store-week rows: 0
Negative demand rows: 0
Infinite numeric values: 0
Missing model values: 0


In [46]:
MODEL_DATA_PATH = (
    PROCESSED_DATA_PATH
    / "final_model_data.csv"
)

model_data.to_csv(
    MODEL_DATA_PATH,
    index=False
)

print(
    "Saved:",
    MODEL_DATA_PATH
)

Saved: /content/pricegraph-edge/data/processed/final_model_data.csv


## 8. Locked 52-Week Train-Test Split

The final evaluation uses the **last 52 unique weekly dates** as an untouched test period.

All observations before this boundary are used for model training. The test period is not used for feature selection, model selection, or hyperparameter tuning.

This chronological split reflects the real business setting: learning from historical demand and predicting a future year.

In [47]:
unique_weeks = np.array(
    sorted(
        model_data["date"]
        .dropna()
        .unique()
    )
)

TEST_WEEKS = 52

if len(unique_weeks) <= TEST_WEEKS:
    raise ValueError(
        "Not enough weekly history to create a 52-week test set."
    )

test_weeks = unique_weeks[-TEST_WEEKS:]

TEST_START_DATE = pd.Timestamp(
    test_weeks[0]
)

TEST_END_DATE = pd.Timestamp(
    test_weeks[-1]
)

print("TEMPORAL SPLIT\n")

print(
    "Total unique weeks:",
    len(unique_weeks)
)

print(
    "Test weeks        :",
    len(test_weeks)
)

print(
    "Test start        :",
    TEST_START_DATE.date()
)

print(
    "Test end          :",
    TEST_END_DATE.date()
)

TEMPORAL SPLIT

Total unique weeks: 226
Test weeks        : 52
Test start        : 2015-05-30
Test end          : 2016-05-21


In [48]:
train_data = (
    model_data[
        model_data["date"]
        < TEST_START_DATE
    ]
    .copy()
    .reset_index(drop=True)
)

test_data = (
    model_data[
        model_data["date"]
        >= TEST_START_DATE
    ]
    .copy()
    .reset_index(drop=True)
)

print("TRAIN DATA\n")

print("Rows:", len(train_data))
print(
    "Date range:",
    train_data["date"].min().date(),
    "to",
    train_data["date"].max().date()
)

print(
    "Unique weeks:",
    train_data["date"].nunique()
)

print("\nTEST DATA\n")

print("Rows:", len(test_data))
print(
    "Date range:",
    test_data["date"].min().date(),
    "to",
    test_data["date"].max().date()
)

print(
    "Unique weeks:",
    test_data["date"].nunique()
)

TRAIN DATA

Rows: 167336
Date range: 2012-01-28 to 2015-05-23
Unique weeks: 174

TEST DATA

Rows: 51729
Date range: 2015-05-30 to 2016-05-21
Unique weeks: 52


In [49]:
train_weeks = set(
    train_data["date"].unique()
)

final_test_weeks = set(
    test_data["date"].unique()
)

overlap = (
    train_weeks
    .intersection(final_test_weeks)
)

print("SPLIT VALIDATION\n")

print(
    "Train/Test week overlap:",
    len(overlap)
)

print(
    "Train ends before test starts:",
    train_data["date"].max()
    <
    test_data["date"].min()
)

print(
    "Final test contains 52 weeks:",
    test_data["date"].nunique()
    == 52
)

assert len(overlap) == 0

assert (
    train_data["date"].max()
    <
    test_data["date"].min()
)

assert (
    test_data["date"].nunique()
    == 52
)

print("\nTemporal split validation: PASSED")

SPLIT VALIDATION

Train/Test week overlap: 0
Train ends before test starts: True
Final test contains 52 weeks: True

Temporal split validation: PASSED


In [50]:
split_summary = pd.DataFrame(
    {
        "Dataset": [
            "Train",
            "Final Test"
        ],

        "Rows": [
            len(train_data),
            len(test_data)
        ],

        "Weeks": [
            train_data["date"].nunique(),
            test_data["date"].nunique()
        ],

        "Mean Units": [
            train_data[TARGET].mean(),
            test_data[TARGET].mean()
        ],

        "Median Units": [
            train_data[TARGET].median(),
            test_data[TARGET].median()
        ]
    }
)

print(
    split_summary
    .round(2)
    .to_string(index=False)
)

   Dataset   Rows  Weeks  Mean Units  Median Units
     Train 167336    174       38.28          18.0
Final Test  51729     52       31.16          15.0


## 9. Champion Model Training

The finalized demand model is a **Poisson XGBoost regressor**, selected through chronological cross-validation in Notebook 01.

The model uses:

- 33 finalized retail features,
- one-hot encoding for categorical variables,
- Poisson regression objective for non-negative count demand,
- the locked hyperparameters selected during model development.

The preprocessing pipeline is fitted **only on the training period**. The untouched 52-week test set is transformed using the fitted training encoder.

In [51]:
champion_params = {
    "n_estimators": 625,
    "learning_rate": 0.035,
    "max_depth": 6,
    "min_child_weight": 5,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.1,
    "reg_lambda": 2.0,
    "objective": "count:poisson",
    "eval_metric": "poisson-nloglik",
    "max_delta_step": 0.7,
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}

print("Champion configuration locked.")

Champion configuration locked.


In [52]:
X_train = train_data[
    champion_features
].copy()

y_train = train_data[
    TARGET
].to_numpy()

X_test = test_data[
    champion_features
].copy()

y_test = test_data[
    TARGET
].to_numpy()

print("Training rows:", len(X_train))
print("Test rows    :", len(X_test))

print(
    "Features     :",
    X_train.shape[1]
)

print(
    "Target minimum:",
    y_train.min()
)

Training rows: 167336
Test rows    : 51729
Features     : 33
Target minimum: 0


In [53]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

X_train_encoded = (
    preprocessor
    .fit_transform(X_train)
)

X_test_encoded = (
    preprocessor
    .transform(X_test)
)

print(
    "Encoded train shape:",
    X_train_encoded.shape
)

print(
    "Encoded test shape :",
    X_test_encoded.shape
)

Encoded train shape: (167336, 145)
Encoded test shape : (51729, 145)


In [54]:
champion_model = XGBRegressor(
    **champion_params
)

champion_model.fit(
    X_train_encoded,
    y_train
)

print("Champion model trained successfully.")

Champion model trained successfully.


In [55]:
champion_predictions = (
    champion_model
    .predict(X_test_encoded)
)

champion_predictions = np.maximum(
    champion_predictions,
    0
)

print(
    "Predictions generated:",
    len(champion_predictions)
)

print(
    "Minimum prediction:",
    round(
        champion_predictions.min(),
        3
    )
)

print(
    "Average prediction:",
    round(
        champion_predictions.mean(),
        3
    )
)

Predictions generated: 51729
Minimum prediction: 0.253
Average prediction: 30.331


In [56]:
assert (
    len(champion_predictions)
    == len(test_data)
)

assert (
    np.isnan(
        champion_predictions
    ).sum()
    == 0
)

assert (
    champion_predictions < 0
).sum() == 0

print("Prediction validation: PASSED")

Prediction validation: PASSED


## 10. Final Evaluation

The finalized Poisson XGBoost model is evaluated on the untouched final 52-week test period.

Performance is compared against a strong retail baseline:

> **Previous Week Baseline:** next week's demand is assumed to equal the previous week's observed demand.

The evaluation uses:

- **MAE — Mean Absolute Error:** average absolute prediction error.
- **RMSE — Root Mean Squared Error:** penalizes larger prediction misses more strongly.
- **WAPE — Weighted Absolute Percentage Error:** total absolute error relative to total actual demand.
- **R² — Coefficient of Determination:** proportion of demand variance explained by the model.

Lower MAE, RMSE, and WAPE are better. Higher R² is better.

In [57]:
def evaluate_predictions(
    actual,
    predicted
):

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    r2 = r2_score(
        actual,
        predicted
    )

    wape = (
        np.abs(
            actual - predicted
        ).sum()
        /
        actual.sum()
    ) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WAPE": wape,
        "R2": r2
    }

In [58]:
baseline_predictions = (
    test_data["lag_1_units"]
    .to_numpy()
)

print(
    "Baseline predictions:",
    len(baseline_predictions)
)

print(
    "Champion predictions:",
    len(champion_predictions)
)

Baseline predictions: 51729
Champion predictions: 51729


In [59]:
baseline_metrics = evaluate_predictions(
    y_test,
    baseline_predictions
)

champion_metrics = evaluate_predictions(
    y_test,
    champion_predictions
)

final_comparison = pd.DataFrame(
    [
        {
            "Model":
                "Previous Week Baseline",
            **baseline_metrics
        },
        {
            "Model":
                "Poisson XGBoost Champion",
            **champion_metrics
        }
    ]
)

print(
    final_comparison
    .round(4)
    .to_string(index=False)
)

                   Model     MAE    RMSE    WAPE     R2
  Previous Week Baseline 10.7879 23.9137 34.6208 0.8165
Poisson XGBoost Champion  9.7208 21.8807 31.1961 0.8464


In [60]:
final_improvement = pd.DataFrame(
    {
        "Metric": [
            "MAE",
            "RMSE",
            "WAPE",
            "R2"
        ],

        "Baseline": [
            baseline_metrics["MAE"],
            baseline_metrics["RMSE"],
            baseline_metrics["WAPE"],
            baseline_metrics["R2"]
        ],

        "Champion": [
            champion_metrics["MAE"],
            champion_metrics["RMSE"],
            champion_metrics["WAPE"],
            champion_metrics["R2"]
        ]
    }
)

final_improvement["Difference"] = (
    final_improvement["Champion"]
    -
    final_improvement["Baseline"]
)

print(
    final_improvement
    .round(4)
    .to_string(index=False)
)

Metric  Baseline  Champion  Difference
   MAE   10.7879    9.7208     -1.0671
  RMSE   23.9137   21.8807     -2.0331
  WAPE   34.6208   31.1961     -3.4246
    R2    0.8165    0.8464      0.0299


In [61]:
mae_improvement_pct = (
    (
        baseline_metrics["MAE"]
        -
        champion_metrics["MAE"]
    )
    /
    baseline_metrics["MAE"]
) * 100

rmse_improvement_pct = (
    (
        baseline_metrics["RMSE"]
        -
        champion_metrics["RMSE"]
    )
    /
    baseline_metrics["RMSE"]
) * 100

wape_improvement_points = (
    baseline_metrics["WAPE"]
    -
    champion_metrics["WAPE"]
)

r2_gain = (
    champion_metrics["R2"]
    -
    baseline_metrics["R2"]
)

print("FINAL MODEL IMPROVEMENT\n")

print(
    f"MAE improvement  : "
    f"{mae_improvement_pct:.2f}%"
)

print(
    f"RMSE improvement : "
    f"{rmse_improvement_pct:.2f}%"
)

print(
    f"WAPE improvement : "
    f"{wape_improvement_points:.2f} percentage points"
)

print(
    f"R² gain          : "
    f"{r2_gain:.4f}"
)

FINAL MODEL IMPROVEMENT

MAE improvement  : 9.89%
RMSE improvement : 8.50%
WAPE improvement : 3.42 percentage points
R² gain          : 0.0299


In [62]:
final_predictions = test_data[
    [
        "date",
        "wm_yr_wk",
        "item_id",
        "dept_id",
        "store_id",
        "state_id",
        "sell_price",
        TARGET
    ]
].copy()

final_predictions[
    "prediction"
] = champion_predictions

final_predictions[
    "baseline_prediction"
] = baseline_predictions

final_predictions[
    "abs_error"
] = (
    final_predictions[TARGET]
    -
    final_predictions["prediction"]
).abs()

final_predictions[
    "signed_error"
] = (
    final_predictions["prediction"]
    -
    final_predictions[TARGET]
)

final_predictions[
    "sq_error"
] = (
    final_predictions[TARGET]
    -
    final_predictions["prediction"]
) ** 2

print(
    final_predictions
    .head(10)
    .round(2)
    .to_string(index=False)
)

      date  wm_yr_wk     item_id dept_id store_id state_id  sell_price  units_sold  prediction  baseline_prediction  abs_error  signed_error  sq_error
2015-05-30     11518 FOODS_1_012 FOODS_1     CA_1       CA        5.56          16   19.750000                 13.0       3.75          3.75     14.06
2015-06-06     11519 FOODS_1_012 FOODS_1     CA_1       CA        5.56          19   21.480000                 16.0       2.48          2.48      6.16
2015-06-13     11520 FOODS_1_012 FOODS_1     CA_1       CA        5.56          25   19.620001                 19.0       5.38         -5.38     28.95
2015-06-20     11521 FOODS_1_012 FOODS_1     CA_1       CA        5.56          19   21.990000                 25.0       2.99          2.99      8.92
2015-06-27     11522 FOODS_1_012 FOODS_1     CA_1       CA        5.56          35   23.240000                 19.0      11.76        -11.76    138.39
2015-07-04     11523 FOODS_1_012 FOODS_1     CA_1       CA        5.56          14   27.940001

In [63]:
FINAL_PREDICTIONS_PATH = (
    PROCESSED_DATA_PATH
    / "final_test_predictions.csv"
)

final_predictions.to_csv(
    FINAL_PREDICTIONS_PATH,
    index=False
)

print(
    "Saved:",
    FINAL_PREDICTIONS_PATH
)

Saved: /content/pricegraph-edge/data/processed/final_test_predictions.csv


## 11. Save Model Artifacts

The finalized model artifacts are saved to the project repository so they can be reused without retraining the model after a Colab runtime restart.

The saved artifacts include:

- trained Poisson XGBoost model,
- fitted categorical preprocessing pipeline,
- finalized feature lists,
- locked model hyperparameters,
- train-test boundary,
- final evaluation metrics.

A reload validation is performed immediately after saving to confirm that the stored model reproduces the original predictions.

In [64]:
import joblib

MODEL_FILE = (
    MODEL_PATH
    / "pricegraph_xgb_poisson.json"
)

PREPROCESSOR_FILE = (
    MODEL_PATH
    / "pricegraph_preprocessor.joblib"
)

METADATA_FILE = (
    MODEL_PATH
    / "pricegraph_model_metadata.json"
)

print("Model artifacts will be saved to:")
print(MODEL_PATH)

Model artifacts will be saved to:
/content/pricegraph-edge/models


In [65]:
champion_model.save_model(
    MODEL_FILE
)

print(
    "Saved model:",
    MODEL_FILE
)

Saved model: /content/pricegraph-edge/models/pricegraph_xgb_poisson.json


In [66]:
joblib.dump(
    preprocessor,
    PREPROCESSOR_FILE
)

print(
    "Saved preprocessor:",
    PREPROCESSOR_FILE
)

Saved preprocessor: /content/pricegraph-edge/models/pricegraph_preprocessor.joblib


In [67]:
model_metadata = {
    "project":
        "PriceGraph Edge",

    "model_type":
        "Poisson XGBoost",

    "target":
        TARGET,

    "feature_count":
        len(champion_features),

    "champion_features":
        champion_features,

    "categorical_features":
        categorical_features,

    "numeric_features":
        numeric_features,

    "model_parameters":
        champion_params,

    "test_weeks":
        int(TEST_WEEKS),

    "test_start_date":
        TEST_START_DATE.strftime(
            "%Y-%m-%d"
        ),

    "test_end_date":
        TEST_END_DATE.strftime(
            "%Y-%m-%d"
        ),

    "final_metrics": {
        "MAE":
            float(
                champion_metrics["MAE"]
            ),

        "RMSE":
            float(
                champion_metrics["RMSE"]
            ),

        "WAPE":
            float(
                champion_metrics["WAPE"]
            ),

        "R2":
            float(
                champion_metrics["R2"]
            )
    },

    "baseline_metrics": {
        "MAE":
            float(
                baseline_metrics["MAE"]
            ),

        "RMSE":
            float(
                baseline_metrics["RMSE"]
            ),

        "WAPE":
            float(
                baseline_metrics["WAPE"]
            ),

        "R2":
            float(
                baseline_metrics["R2"]
            )
    }
}

with open(
    METADATA_FILE,
    "w"
) as file:

    json.dump(
        model_metadata,
        file,
        indent=4
    )

print(
    "Saved metadata:",
    METADATA_FILE
)

Saved metadata: /content/pricegraph-edge/models/pricegraph_model_metadata.json


In [68]:
artifact_files = [
    MODEL_FILE,
    PREPROCESSOR_FILE,
    METADATA_FILE
]

print("ARTIFACT CHECK\n")

for path in artifact_files:

    if not path.exists():
        raise FileNotFoundError(
            f"Artifact was not saved: {path}"
        )

    size_kb = (
        path.stat().st_size
        / 1024
    )

    print(
        f"{path.name:<35}"
        f"{size_kb:>10.2f} KB"
    )

ARTIFACT CHECK

pricegraph_xgb_poisson.json           4508.90 KB
pricegraph_preprocessor.joblib           5.86 KB
pricegraph_model_metadata.json           2.69 KB


In [69]:
loaded_preprocessor = joblib.load(
    PREPROCESSOR_FILE
)

loaded_model = XGBRegressor()

loaded_model.load_model(
    MODEL_FILE
)

with open(
    METADATA_FILE,
    "r"
) as file:

    loaded_metadata = json.load(
        file
    )

print("Artifacts reloaded successfully.")

print(
    "Saved feature count:",
    loaded_metadata[
        "feature_count"
    ]
)

print(
    "Saved model type:",
    loaded_metadata[
        "model_type"
    ]
)

Artifacts reloaded successfully.
Saved feature count: 33
Saved model type: Poisson XGBoost


In [70]:
reload_sample = (
    test_data
    .head(500)
    .copy()
)

reload_encoded = (
    loaded_preprocessor
    .transform(
        reload_sample[
            champion_features
        ]
    )
)

reload_predictions = (
    loaded_model
    .predict(
        reload_encoded
    )
)

reload_predictions = np.maximum(
    reload_predictions,
    0
)

original_sample_predictions = (
    champion_predictions[:500]
)

predictions_match = np.allclose(
    original_sample_predictions,
    reload_predictions,
    rtol=1e-6,
    atol=1e-6
)

print(
    "Reloaded predictions match:",
    predictions_match
)

print(
    "Maximum prediction difference:",
    np.max(
        np.abs(
            original_sample_predictions
            -
            reload_predictions
        )
    )
)

assert predictions_match

print(
    "\nArtifact reload validation: PASSED"
)

Reloaded predictions match: True
Maximum prediction difference: 0.0

Artifact reload validation: PASSED


## 12. Safe Price Scenario Engine

The trained demand model is used to evaluate alternative selling-price scenarios for a selected SKU-store-week context.

To keep the simulation conservative:

- candidate prices must have been observed for the same SKU-store during the **previous 52 weeks**,
- a candidate price must have at least **4 weeks of historical support**,
- the current selling price is always retained,
- only price-dependent features are recalculated,
- all demand-history, event, seasonality, product, and store features remain fixed.

For each candidate price, the model predicts weekly unit demand and estimated revenue:

> **Predicted Revenue = Candidate Price × Predicted Units**

This is a model-based pricing scenario analysis, not a causal estimate of price elasticity.

In [71]:
pricing_context = (
    model_data
    .sort_values(
        [
            "item_id",
            "store_id",
            "date"
        ]
    )
    .reset_index(drop=True)
    .copy()
)

print("PRICING CONTEXT\n")

print(
    "Rows:",
    len(pricing_context)
)

print(
    "Products:",
    pricing_context["item_id"].nunique()
)

print(
    "Stores:",
    pricing_context["store_id"].nunique()
)

print(
    "Date range:",
    pricing_context["date"].min().date(),
    "to",
    pricing_context["date"].max().date()
)

PRICING CONTEXT

Rows: 219065
Products: 100
Stores: 10
Date range: 2012-01-28 to 2016-05-21


In [72]:
def get_scenario_row(
    item_id,
    store_id,
    scenario_date=None
):

    context = pricing_context[
        (pricing_context["item_id"] == item_id)
        &
        (pricing_context["store_id"] == store_id)
    ].copy()

    if context.empty:
        raise ValueError(
            f"No modelling data found for "
            f"{item_id} / {store_id}"
        )

    if scenario_date is None:

        base_row = (
            context
            .sort_values("date")
            .iloc[-1]
            .copy()
        )

    else:

        scenario_date = pd.to_datetime(
            scenario_date
        )

        match = context[
            context["date"] == scenario_date
        ]

        if match.empty:
            raise ValueError(
                f"No modelling row found for "
                f"{item_id} / {store_id} "
                f"on {scenario_date.date()}"
            )

        base_row = (
            match
            .iloc[0]
            .copy()
        )

    return base_row

In [73]:
def get_supported_prices(
    item_id,
    store_id,
    scenario_date,
    current_price,
    lookback_weeks=52,
    min_support_weeks=4
):

    scenario_date = pd.to_datetime(
        scenario_date
    )

    lookback_start = (
        scenario_date
        - pd.Timedelta(
            weeks=lookback_weeks
        )
    )

    history = weekly_data[
        (weekly_data["item_id"] == item_id)
        &
        (weekly_data["store_id"] == store_id)
        &
        (weekly_data["date"] <= scenario_date)
        &
        (weekly_data["date"] >= lookback_start)
    ].copy()

    if history.empty:
        raise ValueError(
            "No historical pricing data found "
            "inside the lookback window."
        )

    support = (
        history
        .groupby(
            "sell_price",
            as_index=False
        )
        .agg(
            weeks_observed=(
                "wm_yr_wk",
                "nunique"
            ),
            first_seen=(
                "date",
                "min"
            ),
            last_seen=(
                "date",
                "max"
            )
        )
        .sort_values("sell_price")
        .reset_index(drop=True)
    )

    support["eligible"] = (
        support["weeks_observed"]
        >= min_support_weeks
    )

    candidate_prices = (
        support.loc[
            support["eligible"],
            "sell_price"
        ]
        .tolist()
    )

    # Current price always remains available
    if not any(
        np.isclose(
            current_price,
            candidate_prices
        )
    ):
        candidate_prices.append(
            current_price
        )

    candidate_prices = sorted(
        set(candidate_prices)
    )

    return support, candidate_prices

In [74]:
def simulate_price_scenarios(
    item_id,
    store_id,
    scenario_date=None,
    lookback_weeks=52,
    min_support_weeks=4
):

    # ---------------------------------
    # 1. Retrieve modelling context
    # ---------------------------------

    base_row = get_scenario_row(
        item_id=item_id,
        store_id=store_id,
        scenario_date=scenario_date
    )

    scenario_date = pd.to_datetime(
        base_row["date"]
    )

    current_price = float(
        base_row["sell_price"]
    )

    # ---------------------------------
    # 2. Find historically supported prices
    # ---------------------------------

    support, candidate_prices = (
        get_supported_prices(
            item_id=item_id,
            store_id=store_id,
            scenario_date=scenario_date,
            current_price=current_price,
            lookback_weeks=lookback_weeks,
            min_support_weeks=min_support_weeks
        )
    )

    # ---------------------------------
    # 3. Create candidate rows
    # ---------------------------------

    scenarios = []

    for candidate_price in candidate_prices:

        row = base_row.copy()

        row["sell_price"] = (
            candidate_price
        )

        row["price_change_pct"] = (
            (
                candidate_price
                - row["previous_price"]
            )
            /
            row["previous_price"]
        ) * 100

        row["price_vs_normal_pct"] = (
            (
                candidate_price
                - row["normal_price_13wk"]
            )
            /
            row["normal_price_13wk"]
        ) * 100

        row["price_vs_hist_mean_pct"] = (
            (
                candidate_price
                - row["hist_mean_price"]
            )
            /
            row["hist_mean_price"]
        ) * 100

        row["price_down_flag"] = int(
            candidate_price
            < row["previous_price"]
        )

        row["price_up_flag"] = int(
            candidate_price
            > row["previous_price"]
        )

        scenarios.append(row)

    scenario_df = pd.DataFrame(
        scenarios
    )

    # ---------------------------------
    # 4. Transform using saved encoder
    # ---------------------------------

    X_scenarios = (
        loaded_preprocessor
        .transform(
            scenario_df[
                champion_features
            ]
        )
    )

    # ---------------------------------
    # 5. Predict weekly demand
    # ---------------------------------

    predicted_units = (
        loaded_model
        .predict(X_scenarios)
    )

    predicted_units = np.maximum(
        predicted_units,
        0
    )

    scenario_df["predicted_units"] = (
        predicted_units
    )

    # ---------------------------------
    # 6. Calculate predicted revenue
    # ---------------------------------

    scenario_df["predicted_revenue"] = (
        scenario_df["sell_price"]
        *
        scenario_df["predicted_units"]
    )

    scenario_df[
        "price_change_vs_current_pct"
    ] = (
        (
            scenario_df["sell_price"]
            - current_price
        )
        /
        current_price
    ) * 100

    scenario_df["is_current_price"] = (
        np.isclose(
            scenario_df["sell_price"],
            current_price
        )
    )

    # ---------------------------------
    # 7. Attach price-support evidence
    # ---------------------------------

    result = scenario_df[
        [
            "date",
            "item_id",
            "store_id",
            "sell_price",
            "price_change_vs_current_pct",
            "predicted_units",
            "predicted_revenue",
            "is_current_price"
        ]
    ].copy()

    result = result.merge(
        support[
            [
                "sell_price",
                "weeks_observed",
                "first_seen",
                "last_seen",
                "eligible"
            ]
        ],
        on="sell_price",
        how="left"
    )

    result["weeks_observed"] = (
        result["weeks_observed"]
        .fillna(0)
        .astype(int)
    )

    result["eligible"] = (
        result["eligible"]
        .fillna(False)
    )

    return (
        result
        .sort_values("sell_price")
        .reset_index(drop=True)
    )

In [75]:
sample_simulation = simulate_price_scenarios(
    item_id="FOODS_3_090",
    store_id="CA_1"
)

print(
    sample_simulation
    .round(2)
    .to_string(index=False)
)

      date     item_id store_id  sell_price  price_change_vs_current_pct  predicted_units  predicted_revenue  is_current_price  weeks_observed first_seen  last_seen  eligible
2016-05-21 FOODS_3_090     CA_1        1.48                        -7.50       449.450012             665.19             False              21 2015-05-23 2015-10-10      True
2016-05-21 FOODS_3_090     CA_1        1.50                        -6.25       457.489990             686.23             False              11 2015-10-17 2015-12-26      True
2016-05-21 FOODS_3_090     CA_1        1.60                         0.00       413.970001             662.35              True              21 2016-01-02 2016-05-21      True


In [76]:
print("SIMULATOR VALIDATION\n")

print(
    "Candidate prices:",
    sample_simulation["sell_price"]
    .tolist()
)

print(
    "Scenario date:",
    sample_simulation["date"]
    .iloc[0]
)

print(
    "Current price:",
    sample_simulation.loc[
        sample_simulation["is_current_price"],
        "sell_price"
    ]
    .iloc[0]
)

print(
    "All predictions non-negative:",
    (
        sample_simulation[
            "predicted_units"
        ] >= 0
    ).all()
)

SIMULATOR VALIDATION

Candidate prices: [1.48, 1.5, 1.6]
Scenario date: 2016-05-21 00:00:00
Current price: 1.6
All predictions non-negative: True


In [77]:
print(
    sample_simulation
    .round(2)
    .to_string(index=False)
)

      date     item_id store_id  sell_price  price_change_vs_current_pct  predicted_units  predicted_revenue  is_current_price  weeks_observed first_seen  last_seen  eligible
2016-05-21 FOODS_3_090     CA_1        1.48                        -7.50       449.450012             665.19             False              21 2015-05-23 2015-10-10      True
2016-05-21 FOODS_3_090     CA_1        1.50                        -6.25       457.489990             686.23             False              11 2015-10-17 2015-12-26      True
2016-05-21 FOODS_3_090     CA_1        1.60                         0.00       413.970001             662.35              True              21 2016-01-02 2016-05-21      True


## 13. Revenue Optimization & Price Recommendation

For each supported price scenario, PriceGraph Edge combines the model's predicted weekly demand with the candidate selling price to estimate revenue.

The candidate with the highest predicted revenue is selected as the model-based price recommendation.

The recommendation includes:

- current selling price,
- recommended selling price,
- predicted demand,
- predicted revenue,
- estimated revenue improvement,
- historical support for the recommended price.

Only prices passing the historical support rules from the scenario engine are considered.

> The recommendation represents a model-based pricing scenario, not a causal guarantee of the demand response.

In [78]:
def recommend_price(
    item_id,
    store_id,
    scenario_date=None,
    lookback_weeks=52,
    min_support_weeks=4
):

    scenarios = simulate_price_scenarios(
        item_id=item_id,
        store_id=store_id,
        scenario_date=scenario_date,
        lookback_weeks=lookback_weeks,
        min_support_weeks=min_support_weeks
    )

    # Current scenario
    current_row = (
        scenarios[
            scenarios["is_current_price"]
        ]
        .iloc[0]
    )

    # Revenue-maximising supported scenario
    best_row = (
        scenarios
        .sort_values(
            "predicted_revenue",
            ascending=False
        )
        .iloc[0]
    )

    current_price = float(
        current_row["sell_price"]
    )

    recommended_price = float(
        best_row["sell_price"]
    )

    current_revenue = float(
        current_row["predicted_revenue"]
    )

    recommended_revenue = float(
        best_row["predicted_revenue"]
    )

    revenue_gain = (
        recommended_revenue
        - current_revenue
    )

    revenue_gain_pct = (
        revenue_gain
        /
        current_revenue
    ) * 100

    price_change_pct = (
        (
            recommended_price
            - current_price
        )
        /
        current_price
    ) * 100

    recommendation = {
        "date":
            best_row["date"],

        "item_id":
            item_id,

        "store_id":
            store_id,

        "current_price":
            current_price,

        "recommended_price":
            recommended_price,

        "price_change_pct":
            price_change_pct,

        "current_predicted_units":
            float(
                current_row[
                    "predicted_units"
                ]
            ),

        "recommended_predicted_units":
            float(
                best_row[
                    "predicted_units"
                ]
            ),

        "current_predicted_revenue":
            current_revenue,

        "recommended_predicted_revenue":
            recommended_revenue,

        "predicted_revenue_gain":
            revenue_gain,

        "predicted_revenue_gain_pct":
            revenue_gain_pct,

        "recommended_price_support_weeks":
            int(
                best_row[
                    "weeks_observed"
                ]
            ),

        "recommendation":
            (
                "HOLD"
                if np.isclose(
                    current_price,
                    recommended_price
                )
                else "CHANGE"
            )
    }

    return recommendation, scenarios

In [79]:
sample_recommendation, sample_scenarios = (
    recommend_price(
        item_id="FOODS_3_090",
        store_id="CA_1"
    )
)

print("PRICE RECOMMENDATION\n")

for key, value in sample_recommendation.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{key:<35}: "
            f"{value:.2f}"
        )

    else:
        print(
            f"{key:<35}: "
            f"{value}"
        )

PRICE RECOMMENDATION

date                               : 2016-05-21 00:00:00
item_id                            : FOODS_3_090
store_id                           : CA_1
current_price                      : 1.60
recommended_price                  : 1.50
price_change_pct                   : -6.25
current_predicted_units            : 413.97
recommended_predicted_units        : 457.49
current_predicted_revenue          : 662.35
recommended_predicted_revenue      : 686.23
predicted_revenue_gain             : 23.88
predicted_revenue_gain_pct         : 3.61
recommended_price_support_weeks    : 11
recommendation                     : CHANGE


In [80]:
recommendation_table = pd.DataFrame(
    [sample_recommendation]
)

print(
    recommendation_table
    .round(2)
    .to_string(index=False)
)

      date     item_id store_id  current_price  recommended_price  price_change_pct  current_predicted_units  recommended_predicted_units  current_predicted_revenue  recommended_predicted_revenue  predicted_revenue_gain  predicted_revenue_gain_pct  recommended_price_support_weeks recommendation
2016-05-21 FOODS_3_090     CA_1            1.6                1.5             -6.25                   413.97                       457.49                     662.35                         686.23                   23.88                        3.61                               11         CHANGE


## 14. Portfolio Pricing Recommendations

The safe pricing engine is applied across the complete portfolio of **100 products × 10 stores**.

For each SKU-store combination, the most recent modelling context is used to:

- identify historically supported candidate prices,
- predict weekly demand under each candidate,
- estimate candidate revenue,
- select the supported price with the highest predicted revenue,
- classify the decision as **HOLD** or **CHANGE**.

The resulting portfolio table provides the main pricing-decision output used later by the Streamlit application.

In [81]:
portfolio_pairs = (
    pricing_context[
        ["item_id", "store_id"]
    ]
    .drop_duplicates()
    .sort_values(
        ["item_id", "store_id"]
    )
    .reset_index(drop=True)
)

print("PORTFOLIO COVERAGE\n")

print(
    "Products:",
    portfolio_pairs["item_id"].nunique()
)

print(
    "Stores:",
    portfolio_pairs["store_id"].nunique()
)

print(
    "SKU-store combinations:",
    len(portfolio_pairs)
)

PORTFOLIO COVERAGE

Products: 100
Stores: 10
SKU-store combinations: 999


In [82]:
expected_pairs = (
    selected_sales_wide[
        ["item_id", "store_id"]
    ]
    .drop_duplicates()
)

actual_pairs = (
    pricing_context[
        ["item_id", "store_id"]
    ]
    .drop_duplicates()
)

missing_pairs = (
    expected_pairs
    .merge(
        actual_pairs,
        on=["item_id", "store_id"],
        how="left",
        indicator=True
    )
    .query('_merge == "left_only"')
    .drop(columns="_merge")
)

print("Missing SKU-store combinations:\n")

print(
    missing_pairs
    .to_string(index=False)
)

Missing SKU-store combinations:

    item_id store_id
FOODS_2_021     CA_2


In [83]:
for _, row in missing_pairs.iterrows():

    item_id = row["item_id"]
    store_id = row["store_id"]

    temp = feature_data[
        (feature_data["item_id"] == item_id)
        &
        (feature_data["store_id"] == store_id)
    ].copy()

    print(
        f"\n{item_id} / {store_id}"
    )

    print("Weekly rows:", len(temp))

    print("\nMissing champion features:")

    missing_counts = (
        temp[champion_features]
        .isna()
        .sum()
    )

    print(
        missing_counts[
            missing_counts > 0
        ]
        .to_string()
    )


FOODS_2_021 / CA_2
Weekly rows: 52

Missing champion features:
price_change_pct           1
price_vs_normal_pct        4
price_vs_hist_mean_pct     4
lag_1_units                1
lag_4_units                4
lag_13_units              13
lag_52_units              52
rolling_4_units            4
rolling_13_units           4
hist_mean_units            4
hist_median_units          4
lag_1_vs_mean              4
lag_4_vs_mean              4
lag_13_vs_mean            13
lag_52_vs_mean            52
rolling_4_vs_mean          4
rolling_13_vs_mean         4


### Portfolio Eligibility

In [85]:
portfolio_eligibility = expected_pairs.copy()

portfolio_eligibility = (
    portfolio_eligibility
    .merge(
        actual_pairs.assign(
            pricing_eligible=True
        ),
        on=[
            "item_id",
            "store_id"
        ],
        how="left"
    )
)

portfolio_eligibility[
    "pricing_eligible"
] = (
    portfolio_eligibility[
        "pricing_eligible"
    ]
    .fillna(False)
)

portfolio_eligibility[
    "eligibility_reason"
] = np.where(
    portfolio_eligibility[
        "pricing_eligible"
    ],
    "Eligible",
    "Insufficient 52-week history"
)

print(
    portfolio_eligibility[
        "eligibility_reason"
    ]
    .value_counts()
    .to_string()
)

eligibility_reason
Eligible                        999
Insufficient 52-week history      1


In [86]:
ELIGIBILITY_PATH = (
    PROCESSED_DATA_PATH
    / "portfolio_pricing_eligibility.csv"
)

portfolio_eligibility.to_csv(
    ELIGIBILITY_PATH,
    index=False
)

print(
    "Saved:",
    ELIGIBILITY_PATH
)

Saved: /content/pricegraph-edge/data/processed/portfolio_pricing_eligibility.csv


In [87]:
portfolio_recommendations = []
portfolio_failures = []

total_pairs = len(portfolio_pairs)

for i, row in portfolio_pairs.iterrows():

    item_id = row["item_id"]
    store_id = row["store_id"]

    try:

        recommendation, _ = recommend_price(
            item_id=item_id,
            store_id=store_id
        )

        portfolio_recommendations.append(
            recommendation
        )

    except Exception as error:

        portfolio_failures.append(
            {
                "item_id": item_id,
                "store_id": store_id,
                "error": str(error)
            }
        )

    if (
        (i + 1) % 100 == 0
        or (i + 1) == total_pairs
    ):

        print(
            f"Processed "
            f"{i + 1}/{total_pairs}"
        )

Processed 100/999
Processed 200/999
Processed 300/999
Processed 400/999
Processed 500/999
Processed 600/999
Processed 700/999
Processed 800/999
Processed 900/999
Processed 999/999


In [88]:
portfolio_results = pd.DataFrame(
    portfolio_recommendations
)

portfolio_failures_df = pd.DataFrame(
    portfolio_failures
)

print("\nPORTFOLIO RUN COMPLETE\n")

print(
    "Successful recommendations:",
    len(portfolio_results)
)

print(
    "Failures:",
    len(portfolio_failures_df)
)


PORTFOLIO RUN COMPLETE

Successful recommendations: 999
Failures: 0


In [89]:
portfolio_metadata = (
    pricing_context
    .sort_values("date")
    .groupby(
        ["item_id", "store_id"],
        as_index=False
    )
    .tail(1)[
        [
            "item_id",
            "store_id",
            "dept_id",
            "state_id"
        ]
    ]
)

portfolio_results = (
    portfolio_results
    .merge(
        portfolio_metadata,
        on=[
            "item_id",
            "store_id"
        ],
        how="left",
        validate="one_to_one"
    )
)

print(
    portfolio_results
    .head()
    .round(2)
    .to_string(index=False)
)

      date     item_id store_id  current_price  recommended_price  price_change_pct  current_predicted_units  recommended_predicted_units  current_predicted_revenue  recommended_predicted_revenue  predicted_revenue_gain  predicted_revenue_gain_pct  recommended_price_support_weeks recommendation dept_id state_id
2016-05-21 FOODS_1_012     CA_1           5.64               5.64              0.00                    20.30                        20.30                     114.51                         114.51                    0.00                        0.00                               25           HOLD FOODS_1       CA
2016-05-21 FOODS_1_012     CA_2           5.64               5.56             -1.42                    51.34                        52.44                     289.57                         291.55                    1.98                        0.68                               26         CHANGE FOODS_1       CA
2016-05-21 FOODS_1_012     CA_3           5.64               

In [90]:
portfolio_results["price_direction"] = np.select(
    [
        portfolio_results["recommended_price"]
        > portfolio_results["current_price"],

        portfolio_results["recommended_price"]
        < portfolio_results["current_price"]
    ],
    [
        "INCREASE",
        "DECREASE"
    ],
    default="HOLD"
)

print(
    portfolio_results["price_direction"]
    .value_counts()
    .to_string()
)

price_direction
HOLD        799
INCREASE    142
DECREASE     58


In [91]:
print("PORTFOLIO PRICING SUMMARY\n")

print(
    "Total decisions:",
    len(portfolio_results)
)

print(
    "HOLD:",
    (
        portfolio_results["price_direction"]
        == "HOLD"
    ).sum()
)

print(
    "DECREASE:",
    (
        portfolio_results["price_direction"]
        == "DECREASE"
    ).sum()
)

print(
    "INCREASE:",
    (
        portfolio_results["price_direction"]
        == "INCREASE"
    ).sum()
)

print(
    "\nAverage predicted revenue gain:",
    round(
        portfolio_results[
            "predicted_revenue_gain_pct"
        ].mean(),
        2
    ),
    "%"
)

print(
    "Median predicted revenue gain:",
    round(
        portfolio_results[
            "predicted_revenue_gain_pct"
        ].median(),
        2
    ),
    "%"
)

PORTFOLIO PRICING SUMMARY

Total decisions: 999
HOLD: 799
DECREASE: 58
INCREASE: 142

Average predicted revenue gain: 9.79 %
Median predicted revenue gain: 0.0 %


In [92]:
top_opportunities = (
    portfolio_results
    .sort_values(
        "predicted_revenue_gain_pct",
        ascending=False
    )
    .head(20)
)

print(
    top_opportunities[
        [
            "item_id",
            "dept_id",
            "store_id",
            "current_price",
            "recommended_price",
            "price_change_pct",
            "current_predicted_units",
            "recommended_predicted_units",
            "current_predicted_revenue",
            "recommended_predicted_revenue",
            "predicted_revenue_gain_pct",
            "recommended_price_support_weeks",
            "price_direction"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

    item_id dept_id store_id  current_price  recommended_price  price_change_pct  current_predicted_units  recommended_predicted_units  current_predicted_revenue  recommended_predicted_revenue  predicted_revenue_gain_pct  recommended_price_support_weeks price_direction
FOODS_1_012 FOODS_1     CA_3           5.64               5.56             -1.42                     1.20                        11.25                       6.76                          62.57                      825.74                               26        DECREASE
FOODS_3_541 FOODS_3     TX_1           0.69               0.94             36.23                     1.73                         9.21                       1.20                           8.66                      623.42                               44        INCREASE
FOODS_3_541 FOODS_3     CA_3           0.88               1.00             13.64                     2.97                        18.47                       2.61                          18.

In [93]:
department_pricing_summary = (
    portfolio_results
    .groupby(
        "dept_id",
        as_index=False
    )
    .agg(
        decisions=(
            "item_id",
            "size"
        ),
        avg_price_change_pct=(
            "price_change_pct",
            "mean"
        ),
        avg_revenue_gain_pct=(
            "predicted_revenue_gain_pct",
            "mean"
        ),
        median_revenue_gain_pct=(
            "predicted_revenue_gain_pct",
            "median"
        )
    )
)

print(
    department_pricing_summary
    .round(2)
    .to_string(index=False)
)

dept_id  decisions  avg_price_change_pct  avg_revenue_gain_pct  median_revenue_gain_pct
FOODS_1        200                  0.18                  6.19                      0.0
FOODS_2        299                  0.06                  0.12                      0.0
FOODS_3        500                  4.09                 17.00                      0.0


In [94]:
store_pricing_summary = (
    portfolio_results
    .groupby(
        "store_id",
        as_index=False
    )
    .agg(
        decisions=(
            "item_id",
            "size"
        ),
        avg_price_change_pct=(
            "price_change_pct",
            "mean"
        ),
        avg_revenue_gain_pct=(
            "predicted_revenue_gain_pct",
            "mean"
        ),
        median_revenue_gain_pct=(
            "predicted_revenue_gain_pct",
            "median"
        )
    )
)

print(
    store_pricing_summary
    .round(2)
    .to_string(index=False)
)

store_id  decisions  avg_price_change_pct  avg_revenue_gain_pct  median_revenue_gain_pct
    CA_1        100                  1.02                  0.83                      0.0
    CA_2         99                  1.76                 26.11                      0.0
    CA_3        100                  2.22                 37.19                      0.0
    CA_4        100                  1.33                  1.07                      0.0
    TX_1        100                  2.43                 16.19                      0.0
    TX_2        100                  3.52                  5.25                      0.0
    TX_3        100                  2.57                  4.52                      0.0
    WI_1        100                  3.15                  1.72                      0.0
    WI_2        100                  1.81                  3.39                      0.0
    WI_3        100                  1.20                  1.75                      0.0


In [95]:
PORTFOLIO_RECOMMENDATIONS_PATH = (
    PROCESSED_DATA_PATH
    / "portfolio_price_recommendations.csv"
)

portfolio_results.to_csv(
    PORTFOLIO_RECOMMENDATIONS_PATH,
    index=False
)

print(
    "Saved:",
    PORTFOLIO_RECOMMENDATIONS_PATH
)

Saved: /content/pricegraph-edge/data/processed/portfolio_price_recommendations.csv


## 15. Portfolio Recommendation Analysis

The portfolio recommendations are reviewed for reliability before being exposed to users.

This section evaluates:

- recommendation direction and magnitude,
- historical support behind recommended prices,
- extreme predicted revenue gains,
- large proposed price movements,
- recommendation reliability using transparent business guardrails.

These checks do not change the demand model. They act as a safety layer around the pricing recommendations.

In [96]:
portfolio_analysis = (
    portfolio_results
    .copy()
    .reset_index(drop=True)
)

portfolio_analysis[
    "absolute_price_change_pct"
] = (
    portfolio_analysis[
        "price_change_pct"
    ]
    .abs()
)

print(
    "Portfolio recommendations:",
    len(portfolio_analysis)
)

Portfolio recommendations: 999


In [97]:
print("RECOMMENDATION DISTRIBUTION\n")

print(
    portfolio_analysis[
        "price_direction"
    ]
    .value_counts()
    .to_string()
)

print(
    "\nPRICE CHANGE DISTRIBUTION\n"
)

print(
    portfolio_analysis[
        "price_change_pct"
    ]
    .describe()
    .round(2)
    .to_string()
)

print(
    "\nPREDICTED REVENUE GAIN DISTRIBUTION\n"
)

print(
    portfolio_analysis[
        "predicted_revenue_gain_pct"
    ]
    .describe()
    .round(2)
    .to_string()
)

RECOMMENDATION DISTRIBUTION

price_direction
HOLD        799
INCREASE    142
DECREASE     58

PRICE CHANGE DISTRIBUTION

count    999.00
mean       2.10
std        7.44
min      -20.00
25%        0.00
50%        0.00
75%        0.00
max       77.60

PREDICTED REVENUE GAIN DISTRIBUTION

count    999.00
mean       9.79
std       63.88
min        0.00
25%        0.00
50%        0.00
75%        0.00
max      825.74


In [98]:
print("RECOMMENDED PRICE SUPPORT\n")

print(
    portfolio_analysis[
        "recommended_price_support_weeks"
    ]
    .describe()
    .round(2)
    .to_string()
)

RECOMMENDED PRICE SUPPORT

count    999.00
mean      39.96
std       16.21
min        2.00
25%       26.00
50%       51.00
75%       53.00
max       53.00


In [99]:
portfolio_analysis[
    "large_price_change_flag"
] = (
    portfolio_analysis[
        "absolute_price_change_pct"
    ] > 15
)

portfolio_analysis[
    "high_revenue_gain_flag"
] = (
    portfolio_analysis[
        "predicted_revenue_gain_pct"
    ] > 20
)

portfolio_analysis[
    "low_support_flag"
] = (
    portfolio_analysis[
        "recommended_price_support_weeks"
    ] < 8
)

print("REVIEW FLAGS\n")

print(
    "Price change > 15%:",
    portfolio_analysis[
        "large_price_change_flag"
    ].sum()
)

print(
    "Revenue gain > 20%:",
    portfolio_analysis[
        "high_revenue_gain_flag"
    ].sum()
)

print(
    "Support < 8 weeks:",
    portfolio_analysis[
        "low_support_flag"
    ].sum()
)

REVIEW FLAGS

Price change > 15%: 50
Revenue gain > 20%: 32
Support < 8 weeks: 43


In [100]:
def assign_reliability(row):

    support = (
        row[
            "recommended_price_support_weeks"
        ]
    )

    price_move = (
        row[
            "absolute_price_change_pct"
        ]
    )

    if (
        support >= 12
        and price_move <= 10
    ):
        return "HIGH"

    elif (
        support >= 8
        and price_move <= 15
    ):
        return "MEDIUM"

    else:
        return "REVIEW"


portfolio_analysis[
    "recommendation_reliability"
] = (
    portfolio_analysis
    .apply(
        assign_reliability,
        axis=1
    )
)

print(
    portfolio_analysis[
        "recommendation_reliability"
    ]
    .value_counts()
    .to_string()
)

recommendation_reliability
HIGH      822
MEDIUM     97
REVIEW     80


In [101]:
review_cases = (
    portfolio_analysis[
        portfolio_analysis[
            "recommendation_reliability"
        ] == "REVIEW"
    ]
    .sort_values(
        "predicted_revenue_gain_pct",
        ascending=False
    )
)

print(
    review_cases[
        [
            "item_id",
            "dept_id",
            "store_id",
            "current_price",
            "recommended_price",
            "price_change_pct",
            "predicted_revenue_gain_pct",
            "recommended_price_support_weeks",
            "price_direction",
            "recommendation_reliability"
        ]
    ]
    .head(25)
    .round(2)
    .to_string(index=False)
)

    item_id dept_id store_id  current_price  recommended_price  price_change_pct  predicted_revenue_gain_pct  recommended_price_support_weeks price_direction recommendation_reliability
FOODS_3_541 FOODS_3     TX_1           0.69               0.94             36.23                      623.42                               44        INCREASE                     REVIEW
FOODS_3_635 FOODS_3     TX_1           0.69               0.94             36.23                      503.30                               42        INCREASE                     REVIEW
FOODS_1_076 FOODS_1     WI_2           6.72               8.72             29.76                      238.27                               10        INCREASE                     REVIEW
FOODS_3_704 FOODS_3     WI_1           2.50               4.44             77.60                       60.57                               52        INCREASE                     REVIEW
FOODS_3_202 FOODS_3     WI_1           2.50               4.44             

In [102]:
top_reliable_opportunities = (
    portfolio_analysis
    .sort_values(
        "predicted_revenue_gain_pct",
        ascending=False
    )
    .head(25)
)

print(
    top_reliable_opportunities[
        [
            "item_id",
            "dept_id",
            "store_id",
            "current_price",
            "recommended_price",
            "price_change_pct",
            "predicted_revenue_gain_pct",
            "recommended_price_support_weeks",
            "recommendation_reliability"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

    item_id dept_id store_id  current_price  recommended_price  price_change_pct  predicted_revenue_gain_pct  recommended_price_support_weeks recommendation_reliability
FOODS_1_012 FOODS_1     CA_3           5.64               5.56             -1.42                      825.74                               26                       HIGH
FOODS_3_541 FOODS_3     TX_1           0.69               0.94             36.23                      623.42                               44                     REVIEW
FOODS_3_541 FOODS_3     CA_3           0.88               1.00             13.64                      606.98                               37                     MEDIUM
FOODS_3_404 FOODS_3     CA_2           0.88               1.00             13.64                      597.74                               46                     MEDIUM
FOODS_3_594 FOODS_3     CA_3           0.88               1.00             13.64                      575.55                               38              

In [103]:
PORTFOLIO_ANALYSIS_PATH = (
    PROCESSED_DATA_PATH
    / "portfolio_price_recommendations_enriched.csv"
)

portfolio_analysis.to_csv(
    PORTFOLIO_ANALYSIS_PATH,
    index=False
)

print(
    "Saved:",
    PORTFOLIO_ANALYSIS_PATH
)

Saved: /content/pricegraph-edge/data/processed/portfolio_price_recommendations_enriched.csv


## 16. Related Product Network

PriceGraph Edge extends single-product pricing analysis by identifying relationships between products within the same department.

To reduce the effect of differences in product scale, weekly demand is normalized relative to each product's recent historical demand.

Pairwise relationships are then measured using correlations between normalized weekly demand movements.

- **Positive correlation** indicates products whose demand tends to move together.
- **Negative correlation** indicates products whose demand tends to move in opposite directions.

These relationships are used to identify candidate product connections for deeper cross-price analysis.

> Correlation alone does not prove substitution, complementarity, or cannibalization.

In [104]:
product_weekly = (
    weekly_data
    .groupby(
        [
            "item_id",
            "dept_id",
            "date"
        ],
        as_index=False
    )
    .agg(
        total_units=(
            "units_sold",
            "sum"
        ),
        avg_price=(
            "sell_price",
            "mean"
        )
    )
    .sort_values(
        [
            "item_id",
            "date"
        ]
    )
    .reset_index(drop=True)
)

print("Rows:", len(product_weekly))
print(
    "Products:",
    product_weekly["item_id"].nunique()
)

Rows: 27730
Products: 100


In [105]:
product_weekly[
    "rolling_13_units"
] = (
    product_weekly
    .groupby("item_id")[
        "total_units"
    ]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            13,
            min_periods=4
        )
        .mean()
    )
)

In [106]:
product_weekly[
    "demand_deviation"
] = (
    (
        product_weekly["total_units"]
        -
        product_weekly["rolling_13_units"]
    )
    /
    product_weekly[
        "rolling_13_units"
    ]
)

product_weekly = (
    product_weekly
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna(
        subset=[
            "demand_deviation"
        ]
    )
    .reset_index(drop=True)
)

print(
    product_weekly[
        "demand_deviation"
    ]
    .describe()
    .round(3)
    .to_string()
)

count    27209.000
mean         0.647
std         19.060
min         -1.000
25%         -0.224
50%         -0.004
75%          0.225
max       1953.333


In [107]:
relationship_rows = []

for dept_id in sorted(
    product_weekly[
        "dept_id"
    ].unique()
):

    dept_data = (
        product_weekly[
            product_weekly[
                "dept_id"
            ] == dept_id
        ]
    )

    demand_matrix = (
        dept_data
        .pivot(
            index="date",
            columns="item_id",
            values="demand_deviation"
        )
    )

    correlation_matrix = (
        demand_matrix
        .corr(
            min_periods=30
        )
    )

    items = (
        correlation_matrix
        .columns
        .tolist()
    )

    for i in range(
        len(items)
    ):

        for j in range(
            i + 1,
            len(items)
        ):

            item_1 = items[i]
            item_2 = items[j]

            correlation = (
                correlation_matrix
                .loc[
                    item_1,
                    item_2
                ]
            )

            if pd.notna(
                correlation
            ):

                relationship_rows.append(
                    {
                        "dept_id":
                            dept_id,

                        "item_1":
                            item_1,

                        "item_2":
                            item_2,

                        "demand_correlation":
                            correlation
                    }
                )

product_relationships = pd.DataFrame(
    relationship_rows
)

print(
    "Product pairs:",
    len(product_relationships)
)

Product pairs: 1850


In [108]:
product_relationships[
    "absolute_correlation"
] = (
    product_relationships[
        "demand_correlation"
    ]
    .abs()
)

product_relationships[
    "relationship_type"
] = np.where(
    product_relationships[
        "demand_correlation"
    ] >= 0,
    "POSITIVE",
    "NEGATIVE"
)

product_relationships = (
    product_relationships
    .sort_values(
        "absolute_correlation",
        ascending=False
    )
    .reset_index(drop=True)
)

In [109]:
print(
    product_relationships[
        [
            "dept_id",
            "item_1",
            "item_2",
            "demand_correlation",
            "relationship_type"
        ]
    ]
    .head(30)
    .round(3)
    .to_string(index=False)
)

dept_id      item_1      item_2  demand_correlation relationship_type
FOODS_3 FOODS_3_090 FOODS_3_491               1.000          POSITIVE
FOODS_3 FOODS_3_090 FOODS_3_723               1.000          POSITIVE
FOODS_3 FOODS_3_030 FOODS_3_224               0.999          POSITIVE
FOODS_3 FOODS_3_318 FOODS_3_723               0.999          POSITIVE
FOODS_3 FOODS_3_295 FOODS_3_501               0.999          POSITIVE
FOODS_3 FOODS_3_295 FOODS_3_406               0.998          POSITIVE
FOODS_3 FOODS_3_406 FOODS_3_501               0.997          POSITIVE
FOODS_3 FOODS_3_090 FOODS_3_738               0.997          POSITIVE
FOODS_3 FOODS_3_133 FOODS_3_339               0.997          POSITIVE
FOODS_2 FOODS_2_128 FOODS_2_360               0.995          POSITIVE
FOODS_2 FOODS_2_021 FOODS_2_360               0.992          POSITIVE
FOODS_3 FOODS_3_727 FOODS_3_822               0.991          POSITIVE
FOODS_3 FOODS_3_324 FOODS_3_785               0.990          POSITIVE
FOODS_3 FOODS_3_257 

In [110]:
negative_relationships = (
    product_relationships[
        product_relationships[
            "demand_correlation"
        ] < 0
    ]
    .sort_values(
        "demand_correlation"
    )
)

print(
    negative_relationships[
        [
            "dept_id",
            "item_1",
            "item_2",
            "demand_correlation"
        ]
    ]
    .head(20)
    .round(3)
    .to_string(index=False)
)

dept_id      item_1      item_2  demand_correlation
FOODS_3 FOODS_3_136 FOODS_3_767              -0.197
FOODS_3 FOODS_3_704 FOODS_3_767              -0.195
FOODS_3 FOODS_3_115 FOODS_3_767              -0.185
FOODS_3 FOODS_3_594 FOODS_3_681              -0.183
FOODS_3 FOODS_3_547 FOODS_3_681              -0.181
FOODS_3 FOODS_3_202 FOODS_3_767              -0.181
FOODS_1 FOODS_1_081 FOODS_1_129              -0.178
FOODS_3 FOODS_3_086 FOODS_3_547              -0.171
FOODS_3 FOODS_3_404 FOODS_3_681              -0.171
FOODS_2 FOODS_2_034 FOODS_2_391              -0.160
FOODS_3 FOODS_3_136 FOODS_3_541              -0.156
FOODS_1 FOODS_1_076 FOODS_1_170              -0.156
FOODS_3 FOODS_3_136 FOODS_3_524              -0.153
FOODS_3 FOODS_3_136 FOODS_3_319              -0.153
FOODS_3 FOODS_3_136 FOODS_3_808              -0.151
FOODS_3 FOODS_3_136 FOODS_3_594              -0.149
FOODS_3 FOODS_3_541 FOODS_3_704              -0.149
FOODS_3 FOODS_3_202 FOODS_3_541              -0.148
FOODS_1 FOOD

In [111]:
RELATIONSHIP_PATH = (
    PROCESSED_DATA_PATH
    / "product_relationships.csv"
)

product_relationships.to_csv(
    RELATIONSHIP_PATH,
    index=False
)

print(
    "Saved:",
    RELATIONSHIP_PATH
)

Saved: /content/pricegraph-edge/data/processed/product_relationships.csv


## 17. Adjusted Product Relationships

Raw product-demand correlations can be inflated by common seasonal, departmental, and market-wide demand movements.

To isolate more product-specific behaviour, the weekly median demand deviation of each department is removed from every product's demand deviation.

The remaining **residual demand movement** represents how a product behaves relative to the broader movement of its own department.

Pairwise correlations are then recalculated using these adjusted demand movements.

> These adjusted relationships are exploratory signals only and do not establish substitution or complementarity.

In [112]:
department_week_effect = (
    product_weekly
    .groupby(
        ["dept_id", "date"],
        as_index=False
    )
    .agg(
        department_demand_deviation=(
            "demand_deviation",
            "median"
        )
    )
)

product_weekly_adjusted = (
    product_weekly
    .merge(
        department_week_effect,
        on=["dept_id", "date"],
        how="left",
        validate="many_to_one"
    )
)

In [113]:
product_weekly_adjusted[
    "residual_demand"
] = (
    product_weekly_adjusted[
        "demand_deviation"
    ]
    -
    product_weekly_adjusted[
        "department_demand_deviation"
    ]
)

print(
    product_weekly_adjusted[
        [
            "demand_deviation",
            "department_demand_deviation",
            "residual_demand"
        ]
    ]
    .describe()
    .round(3)
    .to_string()
)

       demand_deviation  department_demand_deviation  residual_demand
count         27209.000                    27209.000        27209.000
mean              0.647                       -0.005            0.652
std              19.060                        0.158           19.057
min              -1.000                       -0.929           -1.547
25%              -0.224                       -0.083           -0.183
50%              -0.004                       -0.001            0.000
75%               0.225                        0.080            0.202
max            1953.333                        0.547         1953.242


In [114]:
adjusted_relationship_rows = []

for dept_id in sorted(
    product_weekly_adjusted[
        "dept_id"
    ].unique()
):

    dept_data = (
        product_weekly_adjusted[
            product_weekly_adjusted[
                "dept_id"
            ] == dept_id
        ]
    )

    residual_matrix = (
        dept_data
        .pivot(
            index="date",
            columns="item_id",
            values="residual_demand"
        )
    )

    correlation_matrix = (
        residual_matrix
        .corr(
            min_periods=30
        )
    )

    items = (
        correlation_matrix
        .columns
        .tolist()
    )

    for i in range(len(items)):

        for j in range(
            i + 1,
            len(items)
        ):

            item_1 = items[i]
            item_2 = items[j]

            correlation = (
                correlation_matrix
                .loc[
                    item_1,
                    item_2
                ]
            )

            if pd.notna(correlation):

                adjusted_relationship_rows.append(
                    {
                        "dept_id":
                            dept_id,

                        "item_1":
                            item_1,

                        "item_2":
                            item_2,

                        "adjusted_demand_correlation":
                            correlation
                    }
                )

adjusted_relationships = pd.DataFrame(
    adjusted_relationship_rows
)

adjusted_relationships[
    "absolute_correlation"
] = (
    adjusted_relationships[
        "adjusted_demand_correlation"
    ]
    .abs()
)

adjusted_relationships = (
    adjusted_relationships
    .sort_values(
        "absolute_correlation",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Adjusted product pairs:",
    len(adjusted_relationships)
)

Adjusted product pairs: 1850


In [115]:
print(
    adjusted_relationships[
        "adjusted_demand_correlation"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(3)
    .to_string()
)

count    1850.000
mean        0.057
std         0.258
min        -0.298
1%         -0.207
5%         -0.135
10%        -0.105
25%        -0.055
50%        -0.015
75%         0.041
90%         0.213
95%         0.882
99%         0.987
max         1.000


In [116]:
adjusted_positive = (
    adjusted_relationships[
        adjusted_relationships[
            "adjusted_demand_correlation"
        ] > 0
    ]
    .sort_values(
        "adjusted_demand_correlation",
        ascending=False
    )
)

print(
    adjusted_positive[
        [
            "dept_id",
            "item_1",
            "item_2",
            "adjusted_demand_correlation"
        ]
    ]
    .head(25)
    .round(3)
    .to_string(index=False)
)

dept_id      item_1      item_2  adjusted_demand_correlation
FOODS_3 FOODS_3_090 FOODS_3_491                        1.000
FOODS_3 FOODS_3_090 FOODS_3_723                        1.000
FOODS_3 FOODS_3_030 FOODS_3_224                        1.000
FOODS_3 FOODS_3_318 FOODS_3_723                        0.999
FOODS_3 FOODS_3_295 FOODS_3_501                        0.999
FOODS_3 FOODS_3_295 FOODS_3_406                        0.998
FOODS_3 FOODS_3_406 FOODS_3_501                        0.997
FOODS_3 FOODS_3_090 FOODS_3_738                        0.997
FOODS_3 FOODS_3_133 FOODS_3_339                        0.997
FOODS_2 FOODS_2_128 FOODS_2_360                        0.995
FOODS_2 FOODS_2_021 FOODS_2_360                        0.992
FOODS_3 FOODS_3_727 FOODS_3_822                        0.991
FOODS_3 FOODS_3_257 FOODS_3_727                        0.990
FOODS_3 FOODS_3_324 FOODS_3_785                        0.990
FOODS_3 FOODS_3_578 FOODS_3_785                        0.989
FOODS_3 FOODS_3_224 FOOD

In [117]:
adjusted_negative = (
    adjusted_relationships[
        adjusted_relationships[
            "adjusted_demand_correlation"
        ] < 0
    ]
    .sort_values(
        "adjusted_demand_correlation"
    )
)

print(
    adjusted_negative[
        [
            "dept_id",
            "item_1",
            "item_2",
            "adjusted_demand_correlation"
        ]
    ]
    .head(25)
    .round(3)
    .to_string(index=False)
)

dept_id      item_1      item_2  adjusted_demand_correlation
FOODS_2 FOODS_2_019 FOODS_2_285                       -0.298
FOODS_3 FOODS_3_136 FOODS_3_767                       -0.285
FOODS_3 FOODS_3_704 FOODS_3_767                       -0.283
FOODS_2 FOODS_2_197 FOODS_2_285                       -0.278
FOODS_3 FOODS_3_202 FOODS_3_767                       -0.276
FOODS_3 FOODS_3_115 FOODS_3_767                       -0.269
FOODS_1 FOODS_1_076 FOODS_1_170                       -0.255
FOODS_2 FOODS_2_276 FOODS_2_285                       -0.247
FOODS_3 FOODS_3_594 FOODS_3_681                       -0.246
FOODS_2 FOODS_2_274 FOODS_2_285                       -0.234
FOODS_1 FOODS_1_081 FOODS_1_129                       -0.228
FOODS_3 FOODS_3_404 FOODS_3_681                       -0.226
FOODS_3 FOODS_3_136 FOODS_3_541                       -0.218
FOODS_3 FOODS_3_202 FOODS_3_541                       -0.216
FOODS_2 FOODS_2_034 FOODS_2_391                       -0.214
FOODS_3 FOODS_3_136 FOOD

In [118]:
ADJUSTED_RELATIONSHIP_PATH = (
    PROCESSED_DATA_PATH
    / "adjusted_product_relationships.csv"
)

adjusted_relationships.to_csv(
    ADJUSTED_RELATIONSHIP_PATH,
    index=False
)

print(
    "Saved:",
    ADJUSTED_RELATIONSHIP_PATH
)

Saved: /content/pricegraph-edge/data/processed/adjusted_product_relationships.csv


## 18. Cross-Price Screening

Demand correlation alone cannot establish substitution or complementarity.

This section screens candidate product relationships using **cross-price behaviour**.

For each candidate pair, the analysis examines whether a price change in one product is associated with an unusual demand movement in another product within the same store and week.

To reduce confounding:

- demand is normalized relative to each SKU-store's recent 13-week history,
- common department-store-week demand movement is removed,
- only weeks where the source product's price actually changed are used,
- weeks where the target product changed its own price are excluded.

The resulting cross-price signals are exploratory and are not interpreted as causal elasticity estimates.

In [119]:
cross_price_panel = (
    weekly_data
    .sort_values(
        [
            "item_id",
            "store_id",
            "date"
        ]
    )
    .reset_index(drop=True)
    .copy()
)

cross_group = [
    "item_id",
    "store_id"
]

In [120]:
cross_price_panel[
    "rolling_13_store_units"
] = (
    cross_price_panel
    .groupby(cross_group)["units_sold"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            13,
            min_periods=4
        )
        .mean()
    )
)

cross_price_panel[
    "store_demand_deviation"
] = (
    (
        cross_price_panel["units_sold"]
        -
        cross_price_panel[
            "rolling_13_store_units"
        ]
    )
    /
    cross_price_panel[
        "rolling_13_store_units"
    ]
)

In [121]:
cross_price_panel[
    "own_price_change_pct"
] = (
    cross_price_panel
    .groupby(cross_group)["sell_price"]
    .pct_change()
    * 100
)

cross_price_panel[
    "own_price_changed"
] = (
    cross_price_panel[
        "own_price_change_pct"
    ]
    .fillna(0)
    .ne(0)
)

In [122]:
dept_store_week_effect = (
    cross_price_panel
    .groupby(
        [
            "dept_id",
            "store_id",
            "date"
        ],
        as_index=False
    )
    .agg(
        dept_store_demand_deviation=(
            "store_demand_deviation",
            "median"
        )
    )
)

cross_price_panel = (
    cross_price_panel
    .merge(
        dept_store_week_effect,
        on=[
            "dept_id",
            "store_id",
            "date"
        ],
        how="left",
        validate="many_to_one"
    )
)

cross_price_panel[
    "residual_store_demand"
] = (
    cross_price_panel[
        "store_demand_deviation"
    ]
    -
    cross_price_panel[
        "dept_store_demand_deviation"
    ]
)

In [123]:
cross_price_panel = (
    cross_price_panel
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

print(
    cross_price_panel[
        [
            "store_demand_deviation",
            "residual_store_demand",
            "own_price_change_pct"
        ]
    ]
    .describe()
    .round(3)
    .to_string()
)

       store_demand_deviation  residual_store_demand  own_price_change_pct
count              255480.000             255480.000            270065.000
mean                    0.426                  0.474                 0.142
std                     7.854                  7.843                 8.885
min                    -1.000                 -2.376               -96.875
25%                    -0.439                 -0.358                 0.000
50%                    -0.050                  0.000                 0.000
75%                     0.350                  0.374                 0.000
max                  1351.000               1350.952              3100.000


In [124]:
positive_candidates = (
    adjusted_relationships[
        adjusted_relationships[
            "adjusted_demand_correlation"
        ] > 0
    ]
    .nlargest(
        30,
        "adjusted_demand_correlation"
    )
)

negative_candidates = (
    adjusted_relationships[
        adjusted_relationships[
            "adjusted_demand_correlation"
        ] < 0
    ]
    .nsmallest(
        30,
        "adjusted_demand_correlation"
    )
)

candidate_pairs = (
    pd.concat(
        [
            positive_candidates,
            negative_candidates
        ],
        ignore_index=True
    )
    .drop_duplicates(
        [
            "item_1",
            "item_2"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Candidate product pairs:",
    len(candidate_pairs)
)

Candidate product pairs: 60


In [125]:
def evaluate_cross_price_direction(
    target_item,
    source_item,
    dept_id
):

    target = cross_price_panel[
        cross_price_panel["item_id"]
        == target_item
    ][
        [
            "store_id",
            "date",
            "own_price_changed",
            "residual_store_demand"
        ]
    ].copy()

    target = target.rename(
        columns={
            "own_price_changed":
                "target_price_changed",

            "residual_store_demand":
                "target_residual_demand"
        }
    )

    source = cross_price_panel[
        cross_price_panel["item_id"]
        == source_item
    ][
        [
            "store_id",
            "date",
            "own_price_change_pct",
            "own_price_changed"
        ]
    ].copy()

    source = source.rename(
        columns={
            "own_price_change_pct":
                "source_price_change_pct",

            "own_price_changed":
                "source_price_changed"
        }
    )

    paired = target.merge(
        source,
        on=[
            "store_id",
            "date"
        ],
        how="inner"
    )

    # Source price changed,
    # target's own price stayed fixed
    event_data = paired[
        paired["source_price_changed"]
        &
        (~paired["target_price_changed"])
        &
        paired[
            "target_residual_demand"
        ].notna()
        &
        paired[
            "source_price_change_pct"
        ].notna()
    ].copy()

    n_events = len(event_data)

    if n_events < 8:

        return {
            "dept_id":
                dept_id,

            "target_item":
                target_item,

            "source_item":
                source_item,

            "price_change_events":
                n_events,

            "cross_price_corr":
                np.nan,

            "avg_target_dev_source_up":
                np.nan,

            "avg_target_dev_source_down":
                np.nan
        }

    corr = (
        event_data[
            "source_price_change_pct"
        ]
        .corr(
            event_data[
                "target_residual_demand"
            ]
        )
    )

    source_up = event_data[
        event_data[
            "source_price_change_pct"
        ] > 0
    ]

    source_down = event_data[
        event_data[
            "source_price_change_pct"
        ] < 0
    ]

    return {
        "dept_id":
            dept_id,

        "target_item":
            target_item,

        "source_item":
            source_item,

        "price_change_events":
            n_events,

        "cross_price_corr":
            corr,

        "avg_target_dev_source_up":
            (
                source_up[
                    "target_residual_demand"
                ].mean()
                if len(source_up) > 0
                else np.nan
            ),

        "avg_target_dev_source_down":
            (
                source_down[
                    "target_residual_demand"
                ].mean()
                if len(source_down) > 0
                else np.nan
            )
    }

In [126]:
cross_price_results = []

for _, row in candidate_pairs.iterrows():

    item_1 = row["item_1"]
    item_2 = row["item_2"]
    dept_id = row["dept_id"]

    cross_price_results.append(
        evaluate_cross_price_direction(
            target_item=item_1,
            source_item=item_2,
            dept_id=dept_id
        )
    )

    cross_price_results.append(
        evaluate_cross_price_direction(
            target_item=item_2,
            source_item=item_1,
            dept_id=dept_id
        )
    )

cross_price_edges = pd.DataFrame(
    cross_price_results
)

print(
    "Directed relationships tested:",
    len(cross_price_edges)
)

print(
    "Relationships with >= 8 price-change events:",
    cross_price_edges[
        "cross_price_corr"
    ]
    .notna()
    .sum()
)

Directed relationships tested: 120
Relationships with >= 8 price-change events: 114


In [127]:
cross_price_edges[
    "absolute_cross_price_corr"
] = (
    cross_price_edges[
        "cross_price_corr"
    ]
    .abs()
)

cross_price_edges[
    "signal_type"
] = np.select(
    [
        cross_price_edges[
            "cross_price_corr"
        ] >= 0.20,

        cross_price_edges[
            "cross_price_corr"
        ] <= -0.20
    ],
    [
        "SUBSTITUTION_LIKE",
        "COMPLEMENT_LIKE"
    ],
    default="WEAK"
)

In [128]:
strong_cross_price_edges = (
    cross_price_edges[
        cross_price_edges[
            "cross_price_corr"
        ].notna()
    ]
    .sort_values(
        "absolute_cross_price_corr",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    strong_cross_price_edges[
        [
            "dept_id",
            "source_item",
            "target_item",
            "price_change_events",
            "cross_price_corr",
            "avg_target_dev_source_up",
            "avg_target_dev_source_down",
            "signal_type"
        ]
    ]
    .head(30)
    .round(3)
    .to_string(index=False)
)

dept_id source_item target_item  price_change_events  cross_price_corr  avg_target_dev_source_up  avg_target_dev_source_down       signal_type
FOODS_2 FOODS_2_285 FOODS_2_197                   47             0.512                     0.337                      -0.137 SUBSTITUTION_LIKE
FOODS_3 FOODS_3_501 FOODS_3_295                   31             0.499                     0.171                      -0.136 SUBSTITUTION_LIKE
FOODS_3 FOODS_3_090 FOODS_3_491                   41            -0.487                    -0.137                       0.347   COMPLEMENT_LIKE
FOODS_3 FOODS_3_723 FOODS_3_318                  120            -0.449                    -0.098                       0.287   COMPLEMENT_LIKE
FOODS_3 FOODS_3_318 FOODS_3_219                   23             0.445                     0.003                      -0.320 SUBSTITUTION_LIKE
FOODS_3 FOODS_3_295 FOODS_3_501                   26             0.416                     0.247                      -0.245 SUBSTITUTION_LIKE

In [129]:
print("CROSS-PRICE SIGNAL COUNTS\n")

print(
    cross_price_edges[
        "signal_type"
    ]
    .value_counts()
    .to_string()
)

CROSS-PRICE SIGNAL COUNTS

signal_type
WEAK                 93
SUBSTITUTION_LIKE    17
COMPLEMENT_LIKE      10


In [130]:
CROSS_PRICE_PATH = (
    PROCESSED_DATA_PATH
    / "cross_price_relationships.csv"
)

cross_price_edges.to_csv(
    CROSS_PRICE_PATH,
    index=False
)

print(
    "Saved:",
    CROSS_PRICE_PATH
)

Saved: /content/pricegraph-edge/data/processed/cross_price_relationships.csv


## 19. Robust Cross-Price Signals

The initial cross-price screening revealed extreme percentage-based demand deviations caused by very small historical demand baselines.

To reduce the influence of these outliers, demand movement is re-expressed using a logarithmic deviation from recent historical demand.

Price-change events are also inspected and trimmed using empirical percentile boundaries before cross-price correlations are recalculated.

This produces a more robust set of directed product relationships for the final PriceGraph network.

In [131]:
robust_cross_panel = (
    cross_price_panel
    .copy()
)

robust_cross_panel[
    "log_demand_deviation"
] = (
    np.log1p(
        robust_cross_panel["units_sold"]
    )
    -
    np.log1p(
        robust_cross_panel[
            "rolling_13_store_units"
        ]
    )
)

In [132]:
robust_week_effect = (
    robust_cross_panel
    .groupby(
        [
            "dept_id",
            "store_id",
            "date"
        ],
        as_index=False
    )
    .agg(
        department_log_deviation=(
            "log_demand_deviation",
            "median"
        )
    )
)

robust_cross_panel = (
    robust_cross_panel
    .merge(
        robust_week_effect,
        on=[
            "dept_id",
            "store_id",
            "date"
        ],
        how="left",
        validate="many_to_one"
    )
)

robust_cross_panel[
    "robust_residual_demand"
] = (
    robust_cross_panel[
        "log_demand_deviation"
    ]
    -
    robust_cross_panel[
        "department_log_deviation"
    ]
)

In [133]:
nonzero_price_changes = (
    robust_cross_panel.loc[
        robust_cross_panel[
            "own_price_change_pct"
        ].ne(0)
        &
        robust_cross_panel[
            "own_price_change_pct"
        ].notna(),
        "own_price_change_pct"
    ]
)

print("NON-ZERO PRICE CHANGES\n")

print(
    nonzero_price_changes
    .describe(
        percentiles=[
            0.01,
            0.025,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.975,
            0.99
        ]
    )
    .round(2)
    .to_string()
)

print("\nROBUST DEMAND DEVIATION\n")

print(
    robust_cross_panel[
        "robust_residual_demand"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .round(3)
    .to_string()
)

NON-ZERO PRICE CHANGES

count    9132.00
mean        4.21
std        48.14
min       -96.88
1%        -51.00
2.5%      -38.59
5%        -32.43
25%       -10.71
50%         1.48
75%        10.45
95%        48.00
97.5%      62.62
99%       103.44
max      3100.00

ROBUST DEMAND DEVIATION

count    267065.000
mean         -0.216
std           1.071
min          -6.840
1%           -4.029
5%           -2.593
25%          -0.395
50%           0.000
75%           0.302
95%           1.002
99%           1.987
max           7.467


In [134]:
# Empirical price-change boundaries
PRICE_LOWER = (
    nonzero_price_changes
    .quantile(0.025)
)

PRICE_UPPER = (
    nonzero_price_changes
    .quantile(0.975)
)

# Robust demand boundaries
DEMAND_LOWER = (
    robust_cross_panel[
        "robust_residual_demand"
    ]
    .quantile(0.01)
)

DEMAND_UPPER = (
    robust_cross_panel[
        "robust_residual_demand"
    ]
    .quantile(0.99)
)

print("ROBUST BOUNDARIES\n")

print(
    f"Price change: "
    f"{PRICE_LOWER:.2f}% "
    f"to {PRICE_UPPER:.2f}%"
)

print(
    f"Demand residual: "
    f"{DEMAND_LOWER:.3f} "
    f"to {DEMAND_UPPER:.3f}"
)

ROBUST BOUNDARIES

Price change: -38.59% to 62.62%
Demand residual: -4.029 to 1.987


In [135]:
robust_cross_panel[
    "robust_residual_clipped"
] = (
    robust_cross_panel[
        "robust_residual_demand"
    ]
    .clip(
        lower=DEMAND_LOWER,
        upper=DEMAND_UPPER
    )
)

robust_cross_panel[
    "valid_price_change"
] = (
    robust_cross_panel[
        "own_price_change_pct"
    ]
    .between(
        PRICE_LOWER,
        PRICE_UPPER,
        inclusive="both"
    )
)

In [136]:
nonzero_events = (
    robust_cross_panel[
        "own_price_change_pct"
    ]
    .notna()
    &
    robust_cross_panel[
        "own_price_change_pct"
    ]
    .ne(0)
)

valid_events = (
    nonzero_events
    &
    robust_cross_panel[
        "valid_price_change"
    ]
)

print("PRICE EVENT FILTER\n")

print(
    "Original non-zero events:",
    nonzero_events.sum()
)

print(
    "Retained events:",
    valid_events.sum()
)

print(
    "Removed events:",
    (
        nonzero_events.sum()
        -
        valid_events.sum()
    )
)

print(
    "Retention rate:",
    round(
        valid_events.sum()
        /
        nonzero_events.sum()
        * 100,
        2
    ),
    "%"
)

PRICE EVENT FILTER

Original non-zero events: 9132
Retained events: 8674
Removed events: 458
Retention rate: 94.98 %


In [137]:
def evaluate_robust_cross_price(
    target_item,
    source_item,
    dept_id
):

    target = robust_cross_panel[
        robust_cross_panel["item_id"]
        == target_item
    ][
        [
            "store_id",
            "date",
            "own_price_changed",
            "robust_residual_clipped"
        ]
    ].copy()

    target = target.rename(
        columns={
            "own_price_changed":
                "target_price_changed",

            "robust_residual_clipped":
                "target_demand_signal"
        }
    )

    source = robust_cross_panel[
        robust_cross_panel["item_id"]
        == source_item
    ][
        [
            "store_id",
            "date",
            "own_price_change_pct",
            "own_price_changed",
            "valid_price_change"
        ]
    ].copy()

    source = source.rename(
        columns={
            "own_price_change_pct":
                "source_price_change_pct",

            "own_price_changed":
                "source_price_changed",

            "valid_price_change":
                "source_valid_price_change"
        }
    )

    paired = target.merge(
        source,
        on=[
            "store_id",
            "date"
        ],
        how="inner"
    )

    event_data = paired[
        paired["source_price_changed"]
        &
        paired["source_valid_price_change"]
        &
        (~paired["target_price_changed"])
        &
        paired[
            "target_demand_signal"
        ].notna()
    ].copy()

    n_events = len(event_data)

    if n_events < 8:

        return {
            "dept_id": dept_id,
            "source_item": source_item,
            "target_item": target_item,
            "price_change_events": n_events,
            "cross_price_corr": np.nan,
            "target_dev_source_up": np.nan,
            "target_dev_source_down": np.nan
        }

    corr = (
        event_data[
            "source_price_change_pct"
        ]
        .corr(
            event_data[
                "target_demand_signal"
            ]
        )
    )

    source_up = event_data[
        event_data[
            "source_price_change_pct"
        ] > 0
    ]

    source_down = event_data[
        event_data[
            "source_price_change_pct"
        ] < 0
    ]

    return {
        "dept_id":
            dept_id,

        "source_item":
            source_item,

        "target_item":
            target_item,

        "price_change_events":
            n_events,

        "cross_price_corr":
            corr,

        "target_dev_source_up":
            source_up[
                "target_demand_signal"
            ].mean()
            if len(source_up) > 0
            else np.nan,

        "target_dev_source_down":
            source_down[
                "target_demand_signal"
            ].mean()
            if len(source_down) > 0
            else np.nan
    }

In [138]:
robust_results = []

for _, row in candidate_pairs.iterrows():

    item_1 = row["item_1"]
    item_2 = row["item_2"]
    dept_id = row["dept_id"]

    robust_results.append(
        evaluate_robust_cross_price(
            target_item=item_1,
            source_item=item_2,
            dept_id=dept_id
        )
    )

    robust_results.append(
        evaluate_robust_cross_price(
            target_item=item_2,
            source_item=item_1,
            dept_id=dept_id
        )
    )

robust_cross_edges = pd.DataFrame(
    robust_results
)

robust_cross_edges[
    "absolute_cross_price_corr"
] = (
    robust_cross_edges[
        "cross_price_corr"
    ]
    .abs()
)

In [139]:
robust_cross_edges[
    "signal_type"
] = np.select(
    [
        robust_cross_edges[
            "cross_price_corr"
        ] >= 0.20,

        robust_cross_edges[
            "cross_price_corr"
        ] <= -0.20
    ],
    [
        "SUBSTITUTION_LIKE",
        "COMPLEMENT_LIKE"
    ],
    default="WEAK"
)

robust_cross_edges = (
    robust_cross_edges
    .sort_values(
        "absolute_cross_price_corr",
        ascending=False
    )
    .reset_index(drop=True)
)

In [140]:
print(
    robust_cross_edges[
        [
            "dept_id",
            "source_item",
            "target_item",
            "price_change_events",
            "cross_price_corr",
            "target_dev_source_up",
            "target_dev_source_down",
            "signal_type"
        ]
    ]
    .head(30)
    .round(3)
    .to_string(index=False)
)

print("\nROBUST SIGNAL COUNTS\n")

print(
    robust_cross_edges[
        "signal_type"
    ]
    .value_counts()
    .to_string()
)

dept_id source_item target_item  price_change_events  cross_price_corr  target_dev_source_up  target_dev_source_down       signal_type
FOODS_2 FOODS_2_360 FOODS_2_021                    8             0.872                 1.215                   0.215 SUBSTITUTION_LIKE
FOODS_2 FOODS_2_360 FOODS_2_128                    9             0.704                 1.233                  -1.472 SUBSTITUTION_LIKE
FOODS_2 FOODS_2_285 FOODS_2_019                   36             0.574                 0.331                  -0.320 SUBSTITUTION_LIKE
FOODS_2 FOODS_2_285 FOODS_2_197                   45             0.516                 0.261                  -0.391 SUBSTITUTION_LIKE
FOODS_3 FOODS_3_090 FOODS_3_491                   41            -0.510                -0.201                   0.230   COMPLEMENT_LIKE
FOODS_3 FOODS_3_501 FOODS_3_295                   28             0.462                 0.143                  -0.178 SUBSTITUTION_LIKE
FOODS_2 FOODS_2_285 FOODS_2_274                   31   

## 20. Full Cross-Price Network Screening

The robust cross-price methodology is now expanded to every product pair within the same department.

This removes dependence on the earlier demand-correlation candidate selection and allows PriceGraph Edge to discover cross-price relationships directly from historical pricing events.

Each relationship is directional:

**Source Product Price → Target Product Demand**

For reliability, final graph edges must have:

- at least 20 usable source-price change events,
- at least 5 source price increases,
- at least 5 source price decreases,
- absolute robust cross-price correlation of at least 0.20,
- directionally consistent average demand response.

The resulting relationships remain associational rather than causal.

In [141]:
def evaluate_final_cross_price(
    target_item,
    source_item,
    dept_id
):

    target = robust_cross_panel[
        robust_cross_panel["item_id"] == target_item
    ][
        [
            "store_id",
            "date",
            "own_price_changed",
            "robust_residual_clipped"
        ]
    ].copy()

    target = target.rename(
        columns={
            "own_price_changed":
                "target_price_changed",

            "robust_residual_clipped":
                "target_demand_signal"
        }
    )

    source = robust_cross_panel[
        robust_cross_panel["item_id"] == source_item
    ][
        [
            "store_id",
            "date",
            "own_price_change_pct",
            "own_price_changed",
            "valid_price_change"
        ]
    ].copy()

    source = source.rename(
        columns={
            "own_price_change_pct":
                "source_price_change_pct",

            "own_price_changed":
                "source_price_changed",

            "valid_price_change":
                "source_valid_price_change"
        }
    )

    paired = target.merge(
        source,
        on=["store_id", "date"],
        how="inner"
    )

    event_data = paired[
        paired["source_price_changed"]
        &
        paired["source_valid_price_change"]
        &
        (~paired["target_price_changed"])
        &
        paired["target_demand_signal"].notna()
    ].copy()

    source_up = event_data[
        event_data["source_price_change_pct"] > 0
    ]

    source_down = event_data[
        event_data["source_price_change_pct"] < 0
    ]

    n_events = len(event_data)
    n_up = len(source_up)
    n_down = len(source_down)

    if n_events < 8:

        corr = np.nan

    else:

        corr = (
            event_data[
                "source_price_change_pct"
            ]
            .corr(
                event_data[
                    "target_demand_signal"
                ]
            )
        )

    return {
        "dept_id": dept_id,
        "source_item": source_item,
        "target_item": target_item,

        "price_change_events": n_events,
        "source_up_events": n_up,
        "source_down_events": n_down,

        "cross_price_corr": corr,

        "target_dev_source_up":
            source_up[
                "target_demand_signal"
            ].mean()
            if n_up > 0
            else np.nan,

        "target_dev_source_down":
            source_down[
                "target_demand_signal"
            ].mean()
            if n_down > 0
            else np.nan
    }

In [142]:
from itertools import combinations


all_department_pairs = []

for dept_id in sorted(
    robust_cross_panel["dept_id"].unique()
):

    items = sorted(
        robust_cross_panel.loc[
            robust_cross_panel[
                "dept_id"
            ] == dept_id,
            "item_id"
        ]
        .unique()
    )

    for item_1, item_2 in combinations(
        items,
        2
    ):

        all_department_pairs.append(
            {
                "dept_id": dept_id,
                "item_1": item_1,
                "item_2": item_2
            }
        )


all_department_pairs = pd.DataFrame(
    all_department_pairs
)

print(
    "Undirected same-department pairs:",
    len(all_department_pairs)
)

print(
    "Directed relationships to test:",
    len(all_department_pairs) * 2
)

Undirected same-department pairs: 1850
Directed relationships to test: 3700


In [143]:
full_cross_price_results = []

total_pairs = len(
    all_department_pairs
)

for i, row in (
    all_department_pairs.iterrows()
):

    item_1 = row["item_1"]
    item_2 = row["item_2"]
    dept_id = row["dept_id"]

    full_cross_price_results.append(
        evaluate_final_cross_price(
            target_item=item_1,
            source_item=item_2,
            dept_id=dept_id
        )
    )

    full_cross_price_results.append(
        evaluate_final_cross_price(
            target_item=item_2,
            source_item=item_1,
            dept_id=dept_id
        )
    )

    if (
        (i + 1) % 200 == 0
        or
        (i + 1) == total_pairs
    ):

        print(
            f"Processed "
            f"{i + 1}/"
            f"{total_pairs} pairs"
        )


full_cross_edges = pd.DataFrame(
    full_cross_price_results
)

print(
    "\nDirected relationships:",
    len(full_cross_edges)
)

Processed 200/1850 pairs
Processed 400/1850 pairs
Processed 600/1850 pairs
Processed 800/1850 pairs
Processed 1000/1850 pairs
Processed 1200/1850 pairs
Processed 1400/1850 pairs
Processed 1600/1850 pairs
Processed 1800/1850 pairs
Processed 1850/1850 pairs

Directed relationships: 3700


In [144]:
full_cross_edges[
    "absolute_cross_price_corr"
] = (
    full_cross_edges[
        "cross_price_corr"
    ]
    .abs()
)


full_cross_edges[
    "direction_consistent"
] = np.select(
    [
        (
            full_cross_edges[
                "cross_price_corr"
            ] > 0
        ),

        (
            full_cross_edges[
                "cross_price_corr"
            ] < 0
        )
    ],

    [
        (
            full_cross_edges[
                "target_dev_source_up"
            ]
            >
            full_cross_edges[
                "target_dev_source_down"
            ]
        ),

        (
            full_cross_edges[
                "target_dev_source_up"
            ]
            <
            full_cross_edges[
                "target_dev_source_down"
            ]
        )
    ],

    default=False
)

In [145]:
final_edge_mask = (

    (
        full_cross_edges[
            "price_change_events"
        ] >= 20
    )

    &

    (
        full_cross_edges[
            "source_up_events"
        ] >= 5
    )

    &

    (
        full_cross_edges[
            "source_down_events"
        ] >= 5
    )

    &

    (
        full_cross_edges[
            "absolute_cross_price_corr"
        ] >= 0.20
    )

    &

    (
        full_cross_edges[
            "direction_consistent"
        ]
    )
)

In [146]:
pricegraph_edges = (
    full_cross_edges[
        final_edge_mask
    ]
    .copy()
)


pricegraph_edges[
    "relationship_type"
] = np.where(
    pricegraph_edges[
        "cross_price_corr"
    ] > 0,

    "SUBSTITUTION_LIKE",

    "COMPLEMENT_LIKE"
)


pricegraph_edges = (
    pricegraph_edges
    .sort_values(
        "absolute_cross_price_corr",
        ascending=False
    )
    .reset_index(drop=True)
)

In [147]:
print(
    "FINAL PRICEGRAPH EDGES:",
    len(pricegraph_edges)
)

print("\nRELATIONSHIP TYPES\n")

print(
    pricegraph_edges[
        "relationship_type"
    ]
    .value_counts()
    .to_string()
)

print("\nSTRONGEST EDGES\n")

print(
    pricegraph_edges[
        [
            "dept_id",
            "source_item",
            "target_item",
            "price_change_events",
            "source_up_events",
            "source_down_events",
            "cross_price_corr",
            "target_dev_source_up",
            "target_dev_source_down",
            "relationship_type"
        ]
    ]
    .head(30)
    .round(3)
    .to_string(index=False)
)

FINAL PRICEGRAPH EDGES: 828

RELATIONSHIP TYPES

relationship_type
SUBSTITUTION_LIKE    450
COMPLEMENT_LIKE      378

STRONGEST EDGES

dept_id source_item target_item  price_change_events  source_up_events  source_down_events  cross_price_corr  target_dev_source_up  target_dev_source_down relationship_type
FOODS_2 FOODS_2_063 FOODS_2_212                   34                23                  11             0.651                 0.089                  -1.834 SUBSTITUTION_LIKE
FOODS_1 FOODS_1_055 FOODS_1_201                   52                33                  19             0.641                 0.259                  -0.899 SUBSTITUTION_LIKE
FOODS_3 FOODS_3_734 FOODS_3_030                   56                47                   9            -0.628                -0.085                   0.937   COMPLEMENT_LIKE
FOODS_2 FOODS_2_291 FOODS_2_166                   57                34                  23            -0.619                -1.550                  -0.098   COMPLEMENT_LIKE


In [148]:
PRICEGRAPH_EDGE_PATH = (
    PROCESSED_DATA_PATH
    / "pricegraph_edges.csv"
)

pricegraph_edges.to_csv(
    PRICEGRAPH_EDGE_PATH,
    index=False
)

print(
    "\nSaved:",
    PRICEGRAPH_EDGE_PATH
)


Saved: /content/pricegraph-edge/data/processed/pricegraph_edges.csv


## 21. PriceGraph Edge Strength

The robust cross-price screening produces a broad catalogue of supported product relationships.

For visualization and interpretation, each relationship is assigned an evidence tier based on:

- magnitude of the cross-price association,
- number of usable historical price-change events,
- support for both price increases and decreases.

All qualifying relationships remain available in the backend, while stronger relationships are prioritized in the interactive PriceGraph network.

These tiers describe historical evidence strength and should not be interpreted as causal confidence.

In [149]:
pricegraph_all_edges = (
    pricegraph_edges
    .copy()
)

print(
    "All supported edges:",
    len(pricegraph_all_edges)
)

All supported edges: 828


In [150]:
def assign_edge_strength(row):

    corr = row[
        "absolute_cross_price_corr"
    ]

    events = row[
        "price_change_events"
    ]

    up_events = row[
        "source_up_events"
    ]

    down_events = row[
        "source_down_events"
    ]

    if (
        corr >= 0.40
        and events >= 40
        and up_events >= 10
        and down_events >= 10
    ):
        return "STRONG"

    elif (
        corr >= 0.30
        and events >= 30
        and up_events >= 8
        and down_events >= 8
    ):
        return "MODERATE"

    else:
        return "SUPPORTED"


pricegraph_all_edges[
    "edge_strength"
] = (
    pricegraph_all_edges
    .apply(
        assign_edge_strength,
        axis=1
    )
)

print(
    pricegraph_all_edges[
        "edge_strength"
    ]
    .value_counts()
    .to_string()
)

edge_strength
SUPPORTED    502
MODERATE     197
STRONG       129


In [151]:
edge_strength_summary = (
    pricegraph_all_edges
    .groupby(
        [
            "edge_strength",
            "relationship_type"
        ]
    )
    .size()
    .reset_index(
        name="edges"
    )
)

print(
    edge_strength_summary
    .to_string(index=False)
)

edge_strength relationship_type  edges
     MODERATE   COMPLEMENT_LIKE    106
     MODERATE SUBSTITUTION_LIKE     91
       STRONG   COMPLEMENT_LIKE     57
       STRONG SUBSTITUTION_LIKE     72
    SUPPORTED   COMPLEMENT_LIKE    215
    SUPPORTED SUBSTITUTION_LIKE    287


In [152]:
pricegraph_priority_edges = (
    pricegraph_all_edges[
        pricegraph_all_edges[
            "edge_strength"
        ].isin(
            [
                "STRONG",
                "MODERATE"
            ]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Priority graph edges:",
    len(pricegraph_priority_edges)
)

Priority graph edges: 326


In [153]:
print(
    pricegraph_priority_edges[
        [
            "dept_id",
            "source_item",
            "target_item",
            "price_change_events",
            "source_up_events",
            "source_down_events",
            "cross_price_corr",
            "relationship_type",
            "edge_strength"
        ]
    ]
    .head(30)
    .round(3)
    .to_string(index=False)
)

dept_id source_item target_item  price_change_events  source_up_events  source_down_events  cross_price_corr relationship_type edge_strength
FOODS_2 FOODS_2_063 FOODS_2_212                   34                23                  11             0.651 SUBSTITUTION_LIKE      MODERATE
FOODS_1 FOODS_1_055 FOODS_1_201                   52                33                  19             0.641 SUBSTITUTION_LIKE        STRONG
FOODS_3 FOODS_3_734 FOODS_3_030                   56                47                   9            -0.628   COMPLEMENT_LIKE      MODERATE
FOODS_2 FOODS_2_291 FOODS_2_166                   57                34                  23            -0.619   COMPLEMENT_LIKE        STRONG
FOODS_2 FOODS_2_291 FOODS_2_240                   56                33                  23            -0.618   COMPLEMENT_LIKE        STRONG
FOODS_1 FOODS_1_055 FOODS_1_166                   51                32                  19             0.611 SUBSTITUTION_LIKE        STRONG
FOODS_2 FOODS

In [154]:
ALL_EDGES_PATH = (
    PROCESSED_DATA_PATH
    / "pricegraph_all_edges.csv"
)

PRIORITY_EDGES_PATH = (
    PROCESSED_DATA_PATH
    / "pricegraph_priority_edges.csv"
)

pricegraph_all_edges.to_csv(
    ALL_EDGES_PATH,
    index=False
)

pricegraph_priority_edges.to_csv(
    PRIORITY_EDGES_PATH,
    index=False
)

print(
    "Saved:",
    ALL_EDGES_PATH
)

print(
    "Saved:",
    PRIORITY_EDGES_PATH
)

Saved: /content/pricegraph-edge/data/processed/pricegraph_all_edges.csv
Saved: /content/pricegraph-edge/data/processed/pricegraph_priority_edges.csv


## 22. PriceGraph Node Table

Each product in PriceGraph is represented as a graph node.

The node table combines:

- product-level commercial information,
- pricing recommendation summaries,
- incoming and outgoing graph relationships,
- substitution-like and complement-like connections,
- aggregate relationship strength.

This table becomes the primary product-level metadata source for the interactive PriceGraph interface.

In [155]:
product_node_base = (
    weekly_data
    .groupby(
        [
            "item_id",
            "dept_id"
        ],
        as_index=False
    )
    .agg(
        total_units=(
            "units_sold",
            "sum"
        ),
        avg_weekly_units=(
            "units_sold",
            "mean"
        ),
        avg_historical_price=(
            "sell_price",
            "mean"
        ),
        stores_observed=(
            "store_id",
            "nunique"
        ),
        weeks_observed=(
            "wm_yr_wk",
            "nunique"
        )
    )
)

print(
    "Products:",
    product_node_base[
        "item_id"
    ].nunique()
)

Products: 100


In [156]:
product_pricing_summary = (
    portfolio_analysis
    .groupby(
        "item_id",
        as_index=False
    )
    .agg(
        eligible_stores=(
            "store_id",
            "nunique"
        ),

        avg_current_price=(
            "current_price",
            "mean"
        ),

        avg_recommended_price=(
            "recommended_price",
            "mean"
        ),

        avg_price_change_pct=(
            "price_change_pct",
            "mean"
        ),

        avg_predicted_revenue_gain_pct=(
            "predicted_revenue_gain_pct",
            "mean"
        ),

        high_reliability_decisions=(
            "recommendation_reliability",
            lambda x:
            (x == "HIGH").sum()
        ),

        review_decisions=(
            "recommendation_reliability",
            lambda x:
            (x == "REVIEW").sum()
        )
    )
)

In [157]:
outgoing_metrics = (
    pricegraph_priority_edges
    .groupby(
        "source_item",
        as_index=False
    )
    .agg(
        outgoing_edges=(
            "target_item",
            "count"
        ),

        outgoing_strength=(
            "absolute_cross_price_corr",
            "sum"
        ),

        strong_outgoing_edges=(
            "edge_strength",
            lambda x:
            (x == "STRONG").sum()
        ),

        substitution_outgoing=(
            "relationship_type",
            lambda x:
            (
                x == "SUBSTITUTION_LIKE"
            ).sum()
        ),

        complement_outgoing=(
            "relationship_type",
            lambda x:
            (
                x == "COMPLEMENT_LIKE"
            ).sum()
        )
    )
    .rename(
        columns={
            "source_item":
                "item_id"
        }
    )
)

In [158]:
incoming_metrics = (
    pricegraph_priority_edges
    .groupby(
        "target_item",
        as_index=False
    )
    .agg(
        incoming_edges=(
            "source_item",
            "count"
        ),

        incoming_strength=(
            "absolute_cross_price_corr",
            "sum"
        ),

        strong_incoming_edges=(
            "edge_strength",
            lambda x:
            (x == "STRONG").sum()
        ),

        substitution_incoming=(
            "relationship_type",
            lambda x:
            (
                x == "SUBSTITUTION_LIKE"
            ).sum()
        ),

        complement_incoming=(
            "relationship_type",
            lambda x:
            (
                x == "COMPLEMENT_LIKE"
            ).sum()
        )
    )
    .rename(
        columns={
            "target_item":
                "item_id"
        }
    )
)

In [159]:
pricegraph_nodes = (
    product_node_base
    .merge(
        product_pricing_summary,
        on="item_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        outgoing_metrics,
        on="item_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        incoming_metrics,
        on="item_id",
        how="left",
        validate="one_to_one"
    )
)

In [160]:
graph_count_columns = [
    "outgoing_edges",
    "strong_outgoing_edges",
    "substitution_outgoing",
    "complement_outgoing",
    "incoming_edges",
    "strong_incoming_edges",
    "substitution_incoming",
    "complement_incoming"
]

graph_strength_columns = [
    "outgoing_strength",
    "incoming_strength"
]

pricegraph_nodes[
    graph_count_columns
] = (
    pricegraph_nodes[
        graph_count_columns
    ]
    .fillna(0)
    .astype(int)
)

pricegraph_nodes[
    graph_strength_columns
] = (
    pricegraph_nodes[
        graph_strength_columns
    ]
    .fillna(0)
)

In [161]:
pricegraph_nodes[
    "total_connections"
] = (
    pricegraph_nodes[
        "outgoing_edges"
    ]
    +
    pricegraph_nodes[
        "incoming_edges"
    ]
)

pricegraph_nodes[
    "total_relationship_strength"
] = (
    pricegraph_nodes[
        "outgoing_strength"
    ]
    +
    pricegraph_nodes[
        "incoming_strength"
    ]
)

pricegraph_nodes[
    "total_strong_connections"
] = (
    pricegraph_nodes[
        "strong_outgoing_edges"
    ]
    +
    pricegraph_nodes[
        "strong_incoming_edges"
    ]
)

In [162]:
top_graph_products = (
    pricegraph_nodes
    .sort_values(
        [
            "total_relationship_strength",
            "total_connections"
        ],
        ascending=False
    )
    .head(20)
)

print(
    top_graph_products[
        [
            "item_id",
            "dept_id",
            "total_units",
            "avg_current_price",
            "avg_recommended_price",
            "avg_predicted_revenue_gain_pct",
            "outgoing_edges",
            "incoming_edges",
            "total_connections",
            "total_strong_connections",
            "total_relationship_strength"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

    item_id dept_id  total_units  avg_current_price  avg_recommended_price  avg_predicted_revenue_gain_pct  outgoing_edges  incoming_edges  total_connections  total_strong_connections  total_relationship_strength
FOODS_3_231 FOODS_3        67196               5.21                   5.46                            2.90              18               2                 20                        14                         8.78
FOODS_3_020 FOODS_3        60041               0.94                   1.04                            6.30              17               1                 18                        11                         7.30
FOODS_3_753 FOODS_3        58631               5.21                   5.46                            2.99              17               2                 19                         7                         7.20
FOODS_3_734 FOODS_3        99695               1.78                   1.78                            0.00              15               0          

In [163]:
print("PRICEGRAPH NODE SUMMARY\n")

print(
    "Total products:",
    len(pricegraph_nodes)
)

print(
    "Products with priority connections:",
    (
        pricegraph_nodes[
            "total_connections"
        ] > 0
    ).sum()
)

print(
    "Products with no priority connections:",
    (
        pricegraph_nodes[
            "total_connections"
        ] == 0
    ).sum()
)

print(
    "Average connections per product:",
    round(
        pricegraph_nodes[
            "total_connections"
        ].mean(),
        2
    )
)

PRICEGRAPH NODE SUMMARY

Total products: 100
Products with priority connections: 90
Products with no priority connections: 10
Average connections per product: 6.52


In [164]:
PRICEGRAPH_NODES_PATH = (
    PROCESSED_DATA_PATH
    / "pricegraph_nodes.csv"
)

pricegraph_nodes.to_csv(
    PRICEGRAPH_NODES_PATH,
    index=False
)

print(
    "Saved:",
    PRICEGRAPH_NODES_PATH
)

Saved: /content/pricegraph-edge/data/processed/pricegraph_nodes.csv


## 23. Product Neighborhood Table

The directed PriceGraph edge catalogue is converted into a product-centric neighborhood table for the interactive application.

Each product can participate in two types of relationships:

- **Outgoing:** the selected product's historical price movements are associated with changes in another product's demand.
- **Incoming:** another product's historical price movements are associated with changes in the selected product's demand.

The neighborhood table preserves the original directed relationship while providing a simple product-centered structure for Streamlit exploration.

In [165]:
outgoing_neighborhood = (
    pricegraph_priority_edges[
        [
            "dept_id",
            "source_item",
            "target_item",
            "price_change_events",
            "source_up_events",
            "source_down_events",
            "cross_price_corr",
            "absolute_cross_price_corr",
            "relationship_type",
            "edge_strength"
        ]
    ]
    .copy()
)

outgoing_neighborhood = (
    outgoing_neighborhood
    .rename(
        columns={
            "source_item": "item_id",
            "target_item": "neighbor_item"
        }
    )
)

outgoing_neighborhood[
    "edge_direction"
] = "OUTGOING"

outgoing_neighborhood[
    "direction_label"
] = "YOUR PRICE → NEIGHBOR DEMAND"

In [166]:
incoming_neighborhood = (
    pricegraph_priority_edges[
        [
            "dept_id",
            "source_item",
            "target_item",
            "price_change_events",
            "source_up_events",
            "source_down_events",
            "cross_price_corr",
            "absolute_cross_price_corr",
            "relationship_type",
            "edge_strength"
        ]
    ]
    .copy()
)

incoming_neighborhood = (
    incoming_neighborhood
    .rename(
        columns={
            "target_item": "item_id",
            "source_item": "neighbor_item"
        }
    )
)

incoming_neighborhood[
    "edge_direction"
] = "INCOMING"

incoming_neighborhood[
    "direction_label"
] = "NEIGHBOR PRICE → YOUR DEMAND"

In [167]:
pricegraph_neighborhoods = (
    pd.concat(
        [
            outgoing_neighborhood,
            incoming_neighborhood
        ],
        ignore_index=True
    )
)

print(
    "Neighborhood rows:",
    len(pricegraph_neighborhoods)
)

Neighborhood rows: 652


In [168]:
neighbor_metadata = (
    pricegraph_nodes[
        [
            "item_id",
            "total_units",
            "avg_current_price",
            "avg_recommended_price",
            "avg_predicted_revenue_gain_pct",
            "total_connections",
            "total_strong_connections"
        ]
    ]
    .rename(
        columns={
            "item_id":
                "neighbor_item",

            "total_units":
                "neighbor_total_units",

            "avg_current_price":
                "neighbor_current_price",

            "avg_recommended_price":
                "neighbor_recommended_price",

            "avg_predicted_revenue_gain_pct":
                "neighbor_revenue_gain_pct",

            "total_connections":
                "neighbor_connections",

            "total_strong_connections":
                "neighbor_strong_connections"
        }
    )
)

pricegraph_neighborhoods = (
    pricegraph_neighborhoods
    .merge(
        neighbor_metadata,
        on="neighbor_item",
        how="left",
        validate="many_to_one"
    )
)

In [169]:
strength_order = {
    "STRONG": 1,
    "MODERATE": 2
}

pricegraph_neighborhoods[
    "strength_rank"
] = (
    pricegraph_neighborhoods[
        "edge_strength"
    ]
    .map(
        strength_order
    )
)

pricegraph_neighborhoods = (
    pricegraph_neighborhoods
    .sort_values(
        [
            "item_id",
            "strength_rank",
            "absolute_cross_price_corr",
            "price_change_events"
        ],
        ascending=[
            True,
            True,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

In [170]:
pricegraph_neighborhoods[
    "neighbor_rank"
] = (
    pricegraph_neighborhoods
    .groupby(
        "item_id"
    )
    .cumcount()
    + 1
)

In [171]:
print("PRICEGRAPH NEIGHBORHOOD SUMMARY\n")

print(
    "Neighborhood rows:",
    len(pricegraph_neighborhoods)
)

print(
    "Products represented:",
    pricegraph_neighborhoods[
        "item_id"
    ].nunique()
)

print(
    "Outgoing rows:",
    (
        pricegraph_neighborhoods[
            "edge_direction"
        ] == "OUTGOING"
    ).sum()
)

print(
    "Incoming rows:",
    (
        pricegraph_neighborhoods[
            "edge_direction"
        ] == "INCOMING"
    ).sum()
)

PRICEGRAPH NEIGHBORHOOD SUMMARY

Neighborhood rows: 652
Products represented: 90
Outgoing rows: 326
Incoming rows: 326


In [172]:
def get_product_neighborhood(
    item_id,
    max_neighbors=10
):

    neighborhood = (
        pricegraph_neighborhoods[
            pricegraph_neighborhoods[
                "item_id"
            ] == item_id
        ]
        .head(
            max_neighbors
        )
        .copy()
    )

    return neighborhood

In [173]:
sample_neighborhood = (
    get_product_neighborhood(
        item_id="FOODS_3_231",
        max_neighbors=10
    )
)

print(
    sample_neighborhood[
        [
            "item_id",
            "neighbor_item",
            "edge_direction",
            "direction_label",
            "cross_price_corr",
            "relationship_type",
            "edge_strength",
            "price_change_events",
            "neighbor_rank"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

    item_id neighbor_item edge_direction              direction_label  cross_price_corr relationship_type edge_strength  price_change_events  neighbor_rank
FOODS_3_231   FOODS_3_723       OUTGOING YOUR PRICE → NEIGHBOR DEMAND             0.577 SUBSTITUTION_LIKE        STRONG                   95              1
FOODS_3_231   FOODS_3_090       OUTGOING YOUR PRICE → NEIGHBOR DEMAND             0.566 SUBSTITUTION_LIKE        STRONG                   95              2
FOODS_3_231   FOODS_3_295       OUTGOING YOUR PRICE → NEIGHBOR DEMAND             0.552 SUBSTITUTION_LIKE        STRONG                   90              3
FOODS_3_231   FOODS_3_406       OUTGOING YOUR PRICE → NEIGHBOR DEMAND             0.544 SUBSTITUTION_LIKE        STRONG                   89              4
FOODS_3_231   FOODS_3_491       OUTGOING YOUR PRICE → NEIGHBOR DEMAND             0.514 SUBSTITUTION_LIKE        STRONG                   95              5
FOODS_3_231   FOODS_3_501       OUTGOING YOUR PRICE → NEIGHBOR D

In [174]:
PRICEGRAPH_NEIGHBORHOOD_PATH = (
    PROCESSED_DATA_PATH
    / "pricegraph_neighborhoods.csv"
)

pricegraph_neighborhoods.to_csv(
    PRICEGRAPH_NEIGHBORHOOD_PATH,
    index=False
)

print(
    "Saved:",
    PRICEGRAPH_NEIGHBORHOOD_PATH
)

Saved: /content/pricegraph-edge/data/processed/pricegraph_neighborhoods.csv


## 24. Streamlit Data Contract

The modelling and graph layers are consolidated into a simple product-level data interface.

For a selected product, the interface returns:

- product metadata,
- portfolio pricing summary,
- graph connectivity statistics,
- strongest related-product relationships.

This keeps the Streamlit application separate from the modelling pipeline and allows the frontend to consume saved production artifacts directly.

In [175]:
def get_product_payload(
    item_id,
    max_neighbors=10
):

    node = (
        pricegraph_nodes[
            pricegraph_nodes[
                "item_id"
            ] == item_id
        ]
        .copy()
    )

    if node.empty:
        raise ValueError(
            f"Unknown product: {item_id}"
        )

    neighborhood = (
        get_product_neighborhood(
            item_id=item_id,
            max_neighbors=max_neighbors
        )
    )

    return {
        "product":
            node.iloc[0].to_dict(),

        "neighbors":
            neighborhood.to_dict(
                orient="records"
            )
    }

In [176]:
sample_payload = (
    get_product_payload(
        item_id="FOODS_3_231",
        max_neighbors=10
    )
)

print("PRODUCT\n")

print(
    pd.Series(
        sample_payload["product"]
    )[
        [
            "item_id",
            "dept_id",
            "avg_current_price",
            "avg_recommended_price",
            "avg_predicted_revenue_gain_pct",
            "outgoing_edges",
            "incoming_edges",
            "total_connections",
            "total_strong_connections"
        ]
    ]
    .round(2)
    .to_string()
)

print("\nNEIGHBORS\n")

print(
    pd.DataFrame(
        sample_payload["neighbors"]
    )[
        [
            "neighbor_item",
            "edge_direction",
            "relationship_type",
            "edge_strength",
            "cross_price_corr",
            "price_change_events"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

PRODUCT

item_id                           FOODS_3_231
dept_id                               FOODS_3
avg_current_price                       5.212
avg_recommended_price                    5.46
avg_predicted_revenue_gain_pct        2.90243
outgoing_edges                             18
incoming_edges                              2
total_connections                          20
total_strong_connections                   14

NEIGHBORS

neighbor_item edge_direction relationship_type edge_strength  cross_price_corr  price_change_events
  FOODS_3_723       OUTGOING SUBSTITUTION_LIKE        STRONG             0.577                   95
  FOODS_3_090       OUTGOING SUBSTITUTION_LIKE        STRONG             0.566                   95
  FOODS_3_295       OUTGOING SUBSTITUTION_LIKE        STRONG             0.552                   90
  FOODS_3_406       OUTGOING SUBSTITUTION_LIKE        STRONG             0.544                   89
  FOODS_3_491       OUTGOING SUBSTITUTION_LIKE        STRONG     

In [177]:
production_artifacts = {
    "model":
        str(MODEL_FILE),

    "preprocessor":
        str(PREPROCESSOR_FILE),

    "model_metadata":
        str(METADATA_FILE),

    "pricing_recommendations":
        str(PORTFOLIO_ANALYSIS_PATH),

    "pricegraph_nodes":
        str(PRICEGRAPH_NODES_PATH),

    "pricegraph_edges":
        str(PRIORITY_EDGES_PATH),

    "pricegraph_neighborhoods":
        str(PRICEGRAPH_NEIGHBORHOOD_PATH)
}

ARTIFACT_MANIFEST_PATH = (
    OUTPUT_PATH
    / "production_artifacts.json"
)

with open(
    ARTIFACT_MANIFEST_PATH,
    "w"
) as file:

    json.dump(
        production_artifacts,
        file,
        indent=4
    )

print(
    "Saved:",
    ARTIFACT_MANIFEST_PATH
)

Saved: /content/pricegraph-edge/outputs/production_artifacts.json


In [178]:
print("PRICEGRAPH EDGE BACKEND\n")

print(
    "Products:",
    len(pricegraph_nodes)
)

print(
    "Pricing decisions:",
    len(portfolio_analysis)
)

print(
    "Supported graph edges:",
    len(pricegraph_all_edges)
)

print(
    "Priority graph edges:",
    len(pricegraph_priority_edges)
)

print(
    "Neighborhood rows:",
    len(pricegraph_neighborhoods)
)

print(
    "Connected products:",
    (
        pricegraph_nodes[
            "total_connections"
        ] > 0
    ).sum()
)

PRICEGRAPH EDGE BACKEND

Products: 100
Pricing decisions: 999
Supported graph edges: 828
Priority graph edges: 326
Neighborhood rows: 652
Connected products: 90
